# HypeCheck — Signal or Noise?

**Track: AI + Finance** · Veritas AI International High School Hackathon 2026
**Team:** Nathaniel Wong (model) · Aaden Yick (data) · Alastair Chu (demo and write-up) · Malvern College Hong Kong

*This project is a prototype and is not financial advice.*

**Our question.** Does X sentiment add predictive information beyond market history about the
**next-day direction** of five memecoins — DOGE, SHIB, PEPE, BONK and FLOKI?

**What goes in, what comes out.** In: the posts published in the 24 hours before a fixed UTC cutoff,
plus that coin's own price history up to the same moment. Out: a probability that the next UTC day
closes up, placed beside three simple baselines and a market-only model, plus a historical replay
where you can see what actually happened next.

**The headline result is negative, and we report it as measured.** Adding sentiment did **not**
improve the forecasts. Market + X scored a *worse* held-out probability error (Brier
0.2491) than market-only (0.2481), and on the 142 test coin-days that actually had posts it was
clearly worse (42.96% vs 51.41% direction accuracy). The date-block interval for the difference
crosses zero. Below, we show exactly where it breaks and why.

**How the guide would scope it.** The guide's example of a weekend-sized version of *"predict the stock market"* is
*"a test of whether news-headline sentiment helps predict next-day price moves for five stocks, reported honestly either way."*
Our project is exactly that test — a test of whether X-post sentiment helps predict next-day price moves for five memecoins (DOGE, SHIB, PEPE, BONK, FLOKI), reported honestly either way — with the market and the news source swapped.

**Starting point.** Suggested project **4.2.4 News headlines and the stock market** (AI + Finance
brief), which invites *"a different market, time period or news source"*. We changed the market
(five memecoins instead of the Dow Jones), the news source (X posts instead of newspaper headlines),
and the evaluation discipline — a pre-declared selection rule, purged partitions, an ablation, a
date-block uncertainty estimate and a failure analysis.

---

## Run it top to bottom

| Step | What it does | Roughly |
|---|---|---|
| 1 · Setup | Installs the pinned libraries (Colab only) | 1–2 min |
| 2 · Data | Fetches every input from its pinned public source and checks SHA-256 | 1–2 min |
| 3 · Code | Writes `pipeline.py` — the whole experiment, readable | instant |
| 4 · Train | Scores posts with frozen FinTwitBERT and fits the models | 3–10 min CPU |
| 5 · Results | Held-out metrics, baselines, per-coin detail, uncertainty | instant |
| 6 · Failures | The confident calls that were wrong, with the posts behind them | instant |
| 7 · Tests | Leakage, tamper and boundary checks | instant |
| 8 · Write-up | Our submission write-up, form answers and credits | — |

`Runtime → Run all` produces every number below from scratch. No paid API, no account, no credentials.

**Links.** [Dashboard](https://adeviwon.github.io/hypecheck/) · [This notebook on Colab](https://colab.research.google.com/github/adeviwon/hypecheck/blob/main/HypeCheck.ipynb) · [Repository](https://github.com/adeviwon/hypecheck)


In [ ]:
# 1 · Setup — a pinned environment, so a rerun reproduces our measured numbers.
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Python {sys.version.split()[0]} on {sys.platform} · running in Google Colab: {IN_COLAB}")

PINNED = [
    "numpy==2.5.4", "pandas==3.1.1", "pyarrow==26.0.1", "scikit-learn==1.9.1",
    "requests==2.34.2", "transformers==5.23.0", "safetensors==0.8.0",
    "huggingface-hub==2.4.0", "torch==2.15.0+cpu", "setuptools==84.2.1",
]

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *PINNED,
         "--extra-index-url", "https://download.pytorch.org/whl/cpu"],
        check=True,
    )
    print("Installed the pinned libraries (CPU PyTorch build).")

import numpy
import pandas
import sklearn

print(f"numpy {numpy.__version__} · pandas {pandas.__version__} · scikit-learn {sklearn.__version__}")

## 1 · The data — three public sources, pinned

Everything the model ever sees arrives here. Nothing is pulled from a live feed at run time, nothing
is invented, and nothing is forward-filled: a missing day stops the run instead of being papered over.

| # | Source | What we take from it | Pin |
|---|---|---|---|
| 1 | [HF `StephanAkkerman/financial-tweets-crypto`](https://huggingface.co/datasets/StephanAkkerman/financial-tweets-crypto) (MIT) | 104,379 posts with text, UTC timestamps, ticker tags and a retrospective collection time | revision `c4b17a94a667af3da1c9ed5144f998e3a4f29966`, upstream LFS SHA-256 `81b2f7563a44c1a103e9468a83d52319dfeae0fc616422b4a4df90d8d684b960` |
| 2 | [Binance public market data](https://github.com/binance/binance-spot-api-docs) (daily and 1-hour spot bars, USDT pairs) | 1,025 daily bars per coin and 1,040 complete UTC days of hourly bars | fixed 2024-01-01 → 2026-10-21 horizon |
| 3 | Our own derived files, published in the [repository](https://github.com/adeviwon/hypecheck) so this notebook runs anywhere | hourly aggregation, BTC context, and a second independent scoring pass | pinned by SHA-256 in `pipeline.py` |

The price files are the exact bytes our measured run used. The pipeline can also rebuild them live
from Binance's public REST API (`api.binance.com/api/v3/klines`) — that is how they were made — but we
fetch the published copies first, because Binance blocks some regions and some cloud IP ranges, and a
submission that only runs in certain countries is not a submission. Either way the digests are
verified before anything is trained.

**Two ways to get the data.** The cell below downloads each pinned file from the repository and checks
its SHA-256 before anything is trained. If you would rather not depend on that, drag the same files from
the submission bundle into Colab's file browser (left sidebar) before running it: the cell accepts an
uploaded copy when its digest matches the pinned digest, and refuses anything that does not.

**Honest note on sizing.** Only **1,124** of the 104,379 posts survive our eligibility rules
(original posts only, non-empty bounded text, an exact supported ticker, deduplicated, inside the
fixed window). That is small, and we treat it as small everywhere below.


In [ ]:
# 2 · Fetch every input and verify it. Bad bytes stop the run.
import hashlib
from pathlib import Path

import requests

ROOT = Path.cwd()
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)

PINNED_POSTS_URL = ("https://huggingface.co/datasets/StephanAkkerman/financial-tweets-crypto/resolve/"
                    "c4b17a94a667af3da1c9ed5144f998e3a4f29966/data/train.parquet")
PUBLISHED = {
    "market-hourly.csv": "d6819bf4ff46c31c88d62130904e858deb0191125b2603910755feeba8cce003",
    "expanded-daily.csv": "2aab9bbe2c0d6782e04ec7902e3414ad564bc09eb35dccb802c75d45b2324935",
    "independent-post-scores.csv": "bb35778b88edeb9e0b11bc2ac9fba6068bf472614a48445eff447c79965e17a2",
    "DOGE-prices.csv": "afc23a84c958279129a02a5e556db41218bcfaae2a403ff100f88d634f44f2c2",
    "SHIB-prices.csv": "6f65ac812bbcbde740ecf9b34120147fe786999ec67df8631508027a5afb719b",
    "PEPE-prices.csv": "024ecb4a9da72818a06d4b86e7d8f87c3c51359a97dab27962a73d5bd609702f",
    "BONK-prices.csv": "18be2016111fa5fc051ba750ca21fa2c7a883ccdc80986f03ae23069a38fee02",
    "FLOKI-prices.csv": "1348af136631c9b4167ea5656aeae5ff1ea567300ae8492a5e92fb81b314d82e",
    "post-features-anon.csv": "2d71dbc5da4cd3efff466ecbbcd117fce4b79735d747f14dfc238ea55d182e29",
    "hype-lexicon.csv": "598d46cb09ddba8571680b36de37a95b9e02432d82c638f65597a327e42a5895"
}


def sha256(path):
    with open(path, "rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def fetch(name, url, expected, min_bytes=4096):
    """Download once, verify the digest off the wire, and keep the file only if it matches."""
    dest = DATA / name
    if dest.exists() and sha256(dest) == expected:
        return dest, "already present"
    # A copy the reader dragged into the Colab file browser is as good as one we download, provided the
    # bytes are the pinned bytes. This keeps the notebook runnable even if the repository is unavailable.
    for candidate in (ROOT / name,):
        if candidate.exists() and candidate.resolve() != dest.resolve() and sha256(candidate) == expected:
            dest.write_bytes(candidate.read_bytes())
            return dest, 'used the uploaded copy'
    for attempt in range(3):
        try:
            response = requests.get(url, timeout=(10, 180))
            response.raise_for_status()
            break
        except Exception as error:                                    # noqa: BLE001
            if attempt == 2:
                raise RuntimeError(f"Could not fetch {name}: {error}") from error
    if len(response.content) < min_bytes:
        raise ValueError(f"{name} is implausibly small; refusing to use it")
    part = dest.with_suffix(dest.suffix + ".part")
    part.write_bytes(response.content)
    if sha256(part) != expected:
        part.unlink(missing_ok=True)
        raise ValueError(f"{name} SHA-256 does not match the pinned copy; refusing to continue")
    part.replace(dest)
    return dest, "downloaded"


for name, expected in [("posts.parquet", "81b2f7563a44c1a103e9468a83d52319dfeae0fc616422b4a4df90d8d684b960")] + list(PUBLISHED.items()):
    url = PINNED_POSTS_URL if name == "posts.parquet" else f"https://raw.githubusercontent.com/adeviwon/hypecheck/main/data/{name}"
    path, how = fetch(name, url, expected)
    print(f"  ok  {name:32s} {path.stat().st_size / 1e6:9.2f} MB  {sha256(path)[:16]}...  {how}")

missing = [name for name in ("posts.parquet", "market-hourly.csv", "expanded-daily.csv",
                             "independent-post-scores.csv") if not (DATA / name).exists()]
assert not missing, f"missing required inputs: {missing}"
print("\nEvery input verified by SHA-256. Nothing here is invented.")

In [ ]:
# 3 · Look at the data before training on it — the guide's own advice.
import pandas as pd

posts = pd.read_parquet("data/posts.parquet")
print(f"Source posts: {len(posts):,}   columns: {list(posts.columns)}")

sample = posts.loc[posts.tweet_type.eq("tweet"), ["tweet_id", "created_at", "first_posted_at", "text", "tickers"]]
for row in sample.sample(5, random_state=7).to_dict("records"):
    tags = ", ".join(row["tickers"]) if row["tickers"] is not None else "(none)"
    print(f"\n  tagged: {tags}")
    print(f"  posted: {row['created_at']}   collected: {row['first_posted_at']}")
    print(f"  text:   {str(row['text'])[:140]!r}")

print("\nTwo clocks matter. `created_at` is when the post was written; `first_posted_at` is when the"
      "\narchive first saw it. We use the LATER of the two as the moment the information was actually"
      "\navailable — the conservative choice, and the one that makes this a real forecasting task.")

## 2 · How the experiment works

`pipeline.py`, written in full by the next cell, is the entire model. The short version:

**1 · Availability, not just timestamps.** A post counts at cutoff `t` (UTC midnight) only if
`max(created_at, first_posted_at)` falls in `[t-24h, t)`. Retweets and quotes are dropped; exact
duplicate text keeps only its earliest appearance; a post must carry an exact supported ticker
(`$DOGE`, `#SHIB`, `PEPE`, ...). The word "bonk" in ordinary prose is not treated as BONK.

**2 · Frozen sentiment, never fine-tuned.** [FinTwitBERT-sentiment](https://huggingface.co/StephanAkkerman/FinTwitBERT-sentiment)
(`StephanAkkerman`, MIT, revision `da059da3b3bbcb43f9ed1aeb5ae61644010c7e1e`) scores each post bullish / bearish / neutral using
safe tensor weights and no remote code. The daily score is `bullish - bearish`, alongside the post
count, the disagreement (spread across posts) and the mean bullish and bearish shares. The checkpoint
was public before our experiment window opens, so it was not trained on our test period.

**3 · Prices are labels, never features.** For each coin we take continuous Binance USDT daily bars.
Every input is **lagged**: `ret1`, `ret3`, `ret7`, 7-day volatility, prior-day range and volume change
all describe days that had already closed. The label is `close/open - 1 > 0` for the **next** UTC day,
and flat counts as not-up. A missing or duplicated day raises instead of being filled.

**4 · Split by date, with a purge.** Train 2024-03-01 → 2025-09-29 · validation 2025-10-01 →
2026-04-10 · test 2026-04-12 → 2026-10-21, roughly 60/20/20. One adjacent date is dropped at each
boundary so no training window's target reaches into the next partition.

**5 · Pre-declared model choice, three baselines.** Pooled regularised logistic regressions share the
numeric features and a one-hot coin identity; the scaler is fitted on training data only;
`C` is chosen from `(0.01, 0.1, 1.0)` on **validation** Brier; threshold 0.5; no refit on validation
and no test-driven changes. The baselines are the ones the brief asks for — always up, training
majority and previous direction.

**6 · What we added beyond the brief.** A different market and news source; an ablation from
market-only → + attention → + sentiment; a secondary richer-context arm (BTC returns plus intraday
hourly features); a second independent scoring pass from a different checkpoint revision as a
reproducibility check; a circular date-block bootstrap so a 0.001 Brier gap is not read as a
discovery; and per-coin sparse-coverage accounting.

**7 · The rules we hold ourselves to.** No test-set tuning. Every claim traceable to a number in
`docs/report.json`. Failures printed, not hidden.


In [ ]:
%%writefile pipeline.py
"""HypeCheck: reproducible retrospective experiment, never a trading service."""
from __future__ import annotations

import argparse
import hashlib
import json
import os
from pathlib import Path
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path(__file__).resolve().parent
DATASET = "StephanAkkerman/financial-tweets-crypto"
DATASET_REV = "c4b17a94a667af3da1c9ed5144f998e3a4f29966"
DATASET_SHA256 = "81b2f7563a44c1a103e9468a83d52319dfeae0fc616422b4a4df90d8d684b960"
MODEL = "StephanAkkerman/FinTwitBERT-sentiment"
MODEL_REV = "da059da3b3bbcb43f9ed1aeb5ae61644010c7e1e"
# Declared secondary inputs, from the independent HypeCheck data package.
# Both files are pinned by SHA-256 so a swapped or edited copy fails closed.
HOURLY_FILE = "market-hourly.csv"
HOURLY_SHA256 = "d6819bf4ff46c31c88d62130904e858deb0191125b2603910755feeba8cce003"
EXPANDED_FILE = "expanded-daily.csv"
EXPANDED_SHA256 = "2aab9bbe2c0d6782e04ec7902e3414ad564bc09eb35dccb802c75d45b2324935"
INDEPENDENT_FILE = "independent-post-scores.csv"
INDEPENDENT_SHA256 = "bb35778b88edeb9e0b11bc2ac9fba6068bf472614a48445eff447c79965e17a2"
INDEPENDENT_MODEL_REV = "c079da7618e551d182ffdf68ef7aa25cb9a79667"
COINS = ("DOGE", "SHIB", "PEPE", "BONK", "FLOKI")
START = pd.Timestamp("2024-03-01", tz="UTC")  # after this checkpoint was public
END = pd.Timestamp("2026-10-22", tz="UTC")  # exclusive, fixed historical horizon
MARKET = ["ret1", "ret3", "ret7", "vol7", "range1", "volume_change"]
ATTENTION = ["post_count", "has_posts"]
SOCIAL = ATTENTION + ["sentiment", "disagreement", "bullish", "bearish"]
# Every extended value is computed only from bars that closed BEFORE cutoff t.
BTC = ["btc_ret1", "btc_ret7"]
INTRADAY = ["hourly_vol1", "hourly_range1", "last_hour_ret", "taker_ratio1"]
MARKET_EXT = MARKET + BTC + INTRADAY


def atomic_json(path: Path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(value, ensure_ascii=False, allow_nan=False, indent=2)
    tmp = path.with_suffix(path.suffix + ".part")
    tmp.write_text(payload, encoding="utf-8")
    tmp.replace(path)


def digest(path: Path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


def request(url, **kwargs):
    """Bounded public GETs only; never accept an input URL from the web UI."""
    for attempt in range(3):
        r = requests.get(url, timeout=(10, 60), **kwargs)
        if r.status_code == 429 or r.status_code >= 500:
            if attempt < 2:
                time.sleep(2 ** attempt)
                continue
        r.raise_for_status()
        return r
    raise RuntimeError("Public data request failed")


def load_posts():
    path = ROOT / "data" / "posts.parquet"
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        url = f"https://huggingface.co/datasets/{DATASET}/resolve/{DATASET_REV}/data/train.parquet"
        r = request(url)
        if len(r.content) > 50_000_000 or not r.content.startswith(b"PAR1"):
            raise ValueError("Unexpected dataset payload; refusing to parse")
        temp = path.with_suffix(".part")
        temp.write_bytes(r.content)
        temp.replace(path)
    if digest(path) != DATASET_SHA256:
        raise ValueError("Dataset SHA-256 does not match pinned Hugging Face LFS object")
    raw = pd.read_parquet(path)
    required = {"tweet_id", "created_at", "text", "tickers", "tweet_type", "first_posted_at"}
    if not required <= set(raw):
        raise ValueError(f"Missing columns: {required - set(raw)}")
    if raw.tweet_id.isna().any() or raw.tweet_id.duplicated().any():
        raise ValueError("Source has missing or duplicate tweet IDs")
    raw["created_at"] = pd.to_datetime(raw.created_at, utc=True, errors="coerce")
    raw["first_posted_at"] = pd.to_datetime(raw.first_posted_at, utc=True, errors="coerce")
    raw["available_at"] = raw[["created_at", "first_posted_at"]].max(axis=1)
    # A source collection time is conservative availability, not proof of live delivery.
    valid = raw.created_at.notna() & raw.available_at.notna() & raw.text.notna()
    original = raw[valid & raw.tweet_type.eq("tweet")].copy()
    original["text"] = original.text.astype(str).str.strip()
    original = original[original.text.str.len().between(8, 4000)]
    original = original.sort_values("available_at").drop_duplicates("text", keep="first")
    original = original[(original.available_at >= START - pd.Timedelta(days=1)) & (original.available_at < END)]
    original["tickers"] = original.tickers.apply(lambda x: list(x) if isinstance(x, (list, np.ndarray)) else [])
    original = original[original.tickers.apply(lambda xs: bool(set(xs) & set(COINS)))].copy()
    original["date"] = original.available_at.dt.floor("D") + pd.Timedelta(days=1)
    return raw, original


def load_prices(coin):
    if coin not in COINS:
        raise ValueError("Unknown coin")
    path = ROOT / "data" / f"{coin}-prices.csv"
    if path.exists():
        prices = pd.read_csv(path)
        prices["date"] = pd.to_datetime(prices.date, utc=True)
    else:
        rows = []
        cursor = int(pd.Timestamp("2024-01-01", tz="UTC").timestamp() * 1000)
        stop = int(END.timestamp() * 1000)
        for _ in range(10):
            batch = request("https://api.binance.com/api/v3/klines", params={
                "symbol": coin + "USDT", "interval": "1d", "startTime": cursor,
                "endTime": stop - 1, "timeZone": "0", "limit": 1000,
            }).json()
            if not isinstance(batch, list):
                raise ValueError("Invalid price response")
            if not batch:
                break
            rows.extend(batch)
            next_cursor = int(batch[-1][0]) + 86_400_000
            if next_cursor <= cursor:
                raise ValueError("Price pagination did not advance")
            cursor = next_cursor
            if cursor >= stop:
                break
            time.sleep(0.15)
        else:
            raise RuntimeError("Price pagination exceeded bounded range")
        if not rows:
            raise ValueError(f"No prices for {coin}")
        prices = pd.DataFrame({
            "date": pd.to_datetime([r[0] for r in rows], unit="ms", utc=True),
            "open": [float(r[1]) for r in rows], "high": [float(r[2]) for r in rows],
            "low": [float(r[3]) for r in rows], "close": [float(r[4]) for r in rows],
            "volume": [float(r[7]) for r in rows],
        })
        temp = path.with_suffix(".part")
        prices.to_csv(temp, index=False)
        temp.replace(path)
    prices = prices.sort_values("date").reset_index(drop=True)
    if prices.date.duplicated().any() or prices.empty:
        raise ValueError("Invalid price date index")
    if not prices.date.eq(prices.date.dt.floor("D")).all() or prices.date.max() >= END:
        raise ValueError("Price bars must open at UTC midnight within the fixed horizon")
    if not (prices.date.diff().dropna() == pd.Timedelta(days=1)).all():
        raise ValueError(f"Missing daily prices for {coin}; no silent forward filling")
    vals = prices[["open", "high", "low", "close", "volume"]].to_numpy(float)
    if not np.isfinite(vals).all() or (vals[:, :4] <= 0).any() or (vals[:, 4] < 0).any():
        raise ValueError("Non-finite or invalid market data")
    if (prices.high < prices[["open", "close", "low"]].max(axis=1)).any() or (prices.low > prices[["open", "close", "high"]].min(axis=1)).any():
        raise ValueError("Invalid OHLC bounds")
    return prices


def score_posts(posts, batch_size=16):
    """Frozen safe-weights inference; checkpoints are data, never Python pickle."""
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    path = ROOT / "data" / "sentiment.json"
    signature = hashlib.sha256((MODEL_REV + "|max_length=128|" + "|".join(posts.tweet_id.astype(str) + posts.text)).encode()).hexdigest()
    cache = json.loads(path.read_text()) if path.exists() else {"signature": signature, "scores": {}}
    if cache.get("signature") != signature:
        cache = {"signature": signature, "scores": {}}
    remaining = posts[~posts.tweet_id.astype(str).isin(cache["scores"])]
    if not remaining.empty:
        os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
        torch.set_num_threads(4)
        torch.manual_seed(42)
        tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=MODEL_REV, trust_remote_code=False, token=False)
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL, revision=MODEL_REV, trust_remote_code=False, use_safetensors=True, token=False,
        ).eval()
        labels = {str(v).upper(): int(k) for k, v in model.config.id2label.items()}
        if set(labels) != {"NEUTRAL", "BULLISH", "BEARISH"}:
            raise ValueError(f"Unexpected sentiment label mapping: {labels}")
        for start in range(0, len(remaining), batch_size):
            batch = remaining.iloc[start:start + batch_size]
            inputs = tokenizer(batch.text.tolist(), padding=True, truncation=True, max_length=128, return_tensors="pt")
            with torch.inference_mode():
                probs = model(**inputs).logits.softmax(-1).cpu().numpy()
            if not np.isfinite(probs).all() or not np.allclose(probs.sum(axis=1), 1, atol=1e-5):
                raise ValueError("Invalid sentiment probabilities")
            for tid, values in zip(batch.tweet_id.astype(str), probs):
                cache["scores"][tid] = {key.lower(): float(values[idx]) for key, idx in labels.items()}
            atomic_json(path, cache)  # resumable after every batch
            print(f"Sentiment: {min(start + batch_size, len(remaining))}/{len(remaining)} uncached posts", flush=True)
    records = []
    for row in posts.to_dict("records"):
        values = cache["scores"][str(row["tweet_id"])]
        row.update(values)
        row["sentiment"] = values["bullish"] - values["bearish"]
        records.append(row)
    if not records:
        raise ValueError("No eligible posts; cannot claim a sentiment experiment")
    return pd.DataFrame(records)


def _read_pinned(name, sha256):
    """Local pinned copies only. A missing or edited file fails closed."""
    path = ROOT / "data" / name
    if not path.exists():
        return None
    if digest(path) != sha256:
        raise ValueError(f"{name} SHA-256 does not match the pinned copy")
    return path


def load_hourly(coin):
    """Hourly Binance spot bars, aggregated to the completed day before each cutoff."""
    if coin not in COINS:
        raise ValueError("Unknown coin")
    path = _read_pinned(HOURLY_FILE, HOURLY_SHA256)
    if path is None:
        raise ValueError(f"data/{HOURLY_FILE} is required; see README for the pinned source")
    columns = ["coin", "open_time", "open_usdt", "high_usdt", "low_usdt", "close_usdt",
               "volume_usdt", "taker_buy_volume_usdt"]
    bars = pd.read_csv(path, usecols=columns)
    bars = bars[bars.coin.eq(coin)].copy()
    bars["open_time"] = pd.to_datetime(bars.open_time, utc=True, errors="coerce")
    bars = bars.sort_values("open_time").reset_index(drop=True)
    if bars.empty or bars.open_time.isna().any() or bars.open_time.duplicated().any():
        raise ValueError(f"Invalid hourly index for {coin}")
    if not bars.open_time.dt.minute.eq(0).all():
        raise ValueError("Hourly bars must open on the hour")
    if not (bars.open_time.diff().dropna() == pd.Timedelta(hours=1)).all():
        raise ValueError(f"Hourly bars are not contiguous for {coin}; no silent interpolation")
    values = bars[["open_usdt", "high_usdt", "low_usdt", "close_usdt", "volume_usdt",
                   "taker_buy_volume_usdt"]].to_numpy(float)
    if not np.isfinite(values).all() or (values[:, :4] <= 0).any() or (values[:, 4:] < 0).any():
        raise ValueError("Non-finite or invalid hourly market data")
    if (bars.high_usdt < bars[["open_usdt", "close_usdt", "low_usdt"]].max(axis=1)).any():
        raise ValueError("Invalid hourly OHLC bounds")
    if (bars.low_usdt > bars[["open_usdt", "close_usdt", "high_usdt"]].min(axis=1)).any():
        raise ValueError("Invalid hourly OHLC bounds")
    bars["day"] = bars.open_time.dt.floor("D")
    bars["hour_ret"] = np.log(bars.close_usdt / bars.open_usdt)
    grouped = bars.groupby("day")
    daily = pd.DataFrame({
        "hourly_vol1": grouped.hour_ret.std(ddof=1),
        "hourly_range1": (grouped.high_usdt.max() - grouped.low_usdt.min()) / grouped.close_usdt.last(),
        "last_hour_ret": grouped.hour_ret.last(),
        "taker_ratio1": grouped.taker_buy_volume_usdt.sum() / grouped.volume_usdt.sum(),
        "hours": grouped.size(),
    })
    daily = daily[daily.hours.eq(24)].drop(columns="hours")
    if daily.empty:
        raise ValueError(f"No complete UTC days of hourly bars for {coin}")
    # A day's 23:00 bar closes at its own midnight, so it informs the NEXT cutoff.
    daily.index = daily.index + pd.Timedelta(days=1)
    return daily.reset_index(names="date")


def load_btc():
    """BTC/USDT daily context from the pinned independent market file."""
    path = _read_pinned(EXPANDED_FILE, EXPANDED_SHA256)
    if path is None:
        raise ValueError(f"data/{EXPANDED_FILE} is required; see README for the pinned source")
    frame = pd.read_csv(path, usecols=["utc_day", "coin", "open_usdt", "close_usdt", "btc_close_usdt"])
    btc = frame[["utc_day", "btc_close_usdt"]].dropna().drop_duplicates("utc_day").copy()
    btc["date"] = pd.to_datetime(btc.utc_day, utc=True, errors="coerce")
    btc = btc.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    if btc.empty or btc.date.duplicated().any():
        raise ValueError("Invalid BTC date index")
    if not (btc.date.diff().dropna() == pd.Timedelta(days=1)).all():
        raise ValueError("BTC daily series has gaps; no silent forward filling")
    closes = btc.btc_close_usdt.to_numpy(float)
    if not np.isfinite(closes).all() or (closes <= 0).any():
        raise ValueError("Invalid BTC closes")
    btc["btc_ret1"] = btc.btc_close_usdt.pct_change().shift(1)
    btc["btc_ret7"] = btc.btc_close_usdt.pct_change(7).shift(1)
    # Same-provider cross-check: the independent daily file must reproduce our own bars exactly.
    for coin in COINS:
        local = ROOT / "data" / f"{coin}-prices.csv"
        if not local.exists():
            continue
        mine = pd.read_csv(local)
        mine["date"] = pd.to_datetime(mine.date, utc=True)
        theirs = frame[frame.coin.eq(coin)].copy()
        theirs["date"] = pd.to_datetime(theirs.utc_day, utc=True, errors="coerce")
        merged = mine.merge(theirs[["date", "open_usdt", "close_usdt"]], on="date", how="inner")
        if merged.empty:
            raise ValueError(f"No overlapping daily bars to cross-check for {coin}")
        if not np.array_equal(merged.open.to_numpy(float), merged.open_usdt.to_numpy(float)):
            raise ValueError(f"Independent daily opens disagree with our Binance bars for {coin}")
        if not np.array_equal(merged.close.to_numpy(float), merged.close_usdt.to_numpy(float)):
            raise ValueError(f"Independent daily closes disagree with our Binance bars for {coin}")
    return btc[["date", "btc_ret1", "btc_ret7"]]


def independent_check(scored):
    """Agreement with a separately produced FinTwitBERT pass on a different pinned revision."""
    path = _read_pinned(INDEPENDENT_FILE, INDEPENDENT_SHA256)
    if path is None:
        return {"available": False, "reason": f"data/{INDEPENDENT_FILE} is absent"}
    other = pd.read_csv(path, usecols=["post_id", "coin", "p_bearish", "p_neutral", "p_bullish"])
    other["post_id"] = other.post_id.astype(str)
    other = other.drop_duplicates("post_id")
    mine = scored[["tweet_id", "text", "bullish", "neutral", "bearish"]].copy()
    mine["post_id"] = mine.tweet_id.astype(str)
    merged = mine.merge(other, on="post_id", how="inner")
    if merged.empty:
        return {"available": False, "reason": "no shared post IDs"}
    classes = ["bearish", "neutral", "bullish"]
    ours = merged[classes].to_numpy(float).argmax(axis=1)
    theirs = merged[["p_" + name for name in classes]].to_numpy(float).argmax(axis=1)
    ours_net = merged.bullish.to_numpy(float) - merged.bearish.to_numpy(float)
    theirs_net = merged.p_bullish.to_numpy(float) - merged.p_bearish.to_numpy(float)
    delta = np.abs(merged.bullish.to_numpy(float) - merged.p_bullish.to_numpy(float))
    length = merged.text.astype(str).str.len().to_numpy(float)
    short, long = length <= 400, length > 400
    return {"available": True, "our_model_revision": MODEL_REV,
            "independent_model_revision": INDEPENDENT_MODEL_REV,
            "shared_posts": int(len(merged)),
            "class_agreement": float((ours == theirs).mean()),
            "net_sentiment_sign_agreement": float((np.sign(ours_net) == np.sign(theirs_net)).mean()),
            "mean_abs_p_bullish_difference": float(delta.mean()),
            "max_abs_p_bullish_difference": float(delta.max()),
            "share_p_bullish_differing_over_0_25": float((delta > 0.25).mean()),
            "length_diagnostic": {
                "mean_abs_difference_short_text": float(delta[short].mean()) if short.any() else None,
                "mean_abs_difference_long_text": float(delta[long].mean()) if long.any() else None,
                "note": "Disagreement is flat across text-length buckets, so truncation does not explain it.",
            },
            "note": "Independent scoring pass from a different pinned checkpoint revision. This is a reproducibility check, not a ground-truth accuracy measure."}


def market_features(prices):
    out = prices.copy()
    returns = out.close.pct_change()
    out["ret1"] = returns.shift(1)
    out["ret3"] = out.close.pct_change(3).shift(1)
    out["ret7"] = out.close.pct_change(7).shift(1)
    out["vol7"] = returns.rolling(7).std().shift(1)
    out["range1"] = ((out.high - out.low) / out.close).shift(1)
    out["volume_change"] = np.log1p(out.volume).diff().shift(1)
    out["actual_return"] = out.close / out.open - 1
    out["actual_up"] = (out.actual_return > 0).astype(int)
    return out


def build_dataset(posts, prices, hourly, btc):
    frames = []
    examples = {}
    for coin in COINS:
        subset = posts[posts.tickers.apply(lambda xs: coin in xs)]
        group = subset.groupby("date")
        daily = group.agg(post_count=("tweet_id", "size"), sentiment=("sentiment", "mean"),
                          disagreement=("sentiment", "std"), bullish=("bullish", "mean"), bearish=("bearish", "mean"))
        daily["disagreement"] = daily.disagreement.fillna(0)
        frame = (market_features(prices[coin]).set_index("date").join(daily)
                 .join(hourly[coin].set_index("date")).join(btc.set_index("date")).reset_index())
        frame["post_count"] = frame.post_count.fillna(0).astype(int)
        frame["has_posts"] = (frame.post_count > 0).astype(int)
        # Zero is an imputation only; has_posts distinguishes unobserved text from neutral.
        frame[SOCIAL] = frame[SOCIAL].fillna(0)
        frame["coin"] = coin
        frame = frame[(frame.date >= START) & (frame.date < END)].dropna(subset=MARKET)
        # The primary row set stays exactly as published; extended inputs must be complete.
        if not np.isfinite(frame[MARKET_EXT].to_numpy(float)).all():
            raise ValueError(f"Extended market features are incomplete for {coin}; refusing to impute")
        for date, samples in group:
            examples[(coin, date.strftime("%Y-%m-%d"))] = [{
                "text": r["text"][:700], "url": r["url"] if isinstance(r.get("url"), str) and r["url"].startswith(("https://twitter.com/", "https://x.com/")) else "",
                "created_at": r["created_at"].isoformat(), "available_at": r["available_at"].isoformat(),
            } for r in samples.sort_values("available_at").tail(3).to_dict("records")]
        frames.append(frame)
    dataset = pd.concat(frames).sort_values(["date", "coin"]).reset_index(drop=True)
    if dataset.duplicated(["date", "coin"]).any() or not np.isfinite(dataset[MARKET_EXT + SOCIAL].to_numpy()).all():
        raise ValueError("Invalid feature table")
    return dataset, examples


def split_dates(frame):
    dates = sorted(frame.date.unique())
    if len(dates) < 150:
        raise ValueError("Too few unique dates for the planned evaluation")
    val_start = pd.Timestamp(dates[int(len(dates) * 0.6)])
    test_start = pd.Timestamp(dates[int(len(dates) * 0.8)])
    # Exclude the adjacent day so a full target window ends BEFORE next partition.
    train = frame[frame.date + pd.Timedelta(days=1) < val_start].copy()
    val = frame[(frame.date >= val_start) & (frame.date + pd.Timedelta(days=1) < test_start)].copy()
    test = frame[frame.date >= test_start].copy()
    for part in (train, val, test):
        if part.actual_up.nunique() != 2 or set(part.coin) != set(COINS):
            raise ValueError("Each partition must cover five coins and both outcomes")
    return train, val, test


def classifier(features, c):
    prep = ColumnTransformer([
        ("numeric", StandardScaler(), features),
        ("coin", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["coin"]),
    ])
    return Pipeline([("prep", prep), ("model", LogisticRegression(C=c, max_iter=1000, random_state=42))])


def scores(y, prob):
    prob = np.asarray(prob)
    if len(y) == 0:
        return {"accuracy": None, "balanced_accuracy": None, "brier": None, "n": 0}
    if not np.isfinite(prob).all() or (prob < 0).any() or (prob > 1).any():
        raise ValueError("Invalid forecast probabilities")
    return {"accuracy": float(accuracy_score(y, prob >= 0.5)),
            "balanced_accuracy": float(balanced_accuracy_score(y, prob >= 0.5)),
            "brier": float(brier_score_loss(y, prob)), "n": int(len(y))}


def metric(name, frame, prob):
    out = {"name": name, **scores(frame.actual_up, prob), "n_dates": int(frame.date.nunique()), "per_coin": {}}
    for coin in COINS:
        mask = frame.coin.eq(coin).to_numpy()
        out["per_coin"][coin] = scores(frame.loc[mask, "actual_up"], np.asarray(prob)[mask])
    return out


def bootstrap_delta(test, market, social):
    # ponytail: 7-day moving blocks, exploratory interval; not a formal edge test.
    tmp = pd.DataFrame({"date": test.date, "delta": (social - test.actual_up.to_numpy()) ** 2 - (market - test.actual_up.to_numpy()) ** 2})
    series = tmp.groupby("date").delta.mean().to_numpy()
    rng = np.random.default_rng(42)
    draws = []
    for _ in range(1000):
        starts = rng.integers(0, len(series), size=int(np.ceil(len(series) / 7)))
        sample = np.concatenate([series[(start + np.arange(7)) % len(series)] for start in starts])[:len(series)]
        draws.append(float(sample.mean()))
    return {"mean_social_minus_market_brier": float(series.mean()), "interval95": [float(x) for x in np.quantile(draws, [0.025, 0.975])],
            "method": "1000 circular moving-block resamples of UTC dates, 7-day blocks; coins grouped by day; exploratory interval"}


def fit_and_report(raw, posts, prices, hourly, btc):
    frame, examples = build_dataset(posts, prices, hourly, btc)
    train, val, test = split_dates(frame)
    configurations = {"Market only": MARKET, "Market + attention": MARKET + ATTENTION, "Market + X": MARKET + SOCIAL}
    models, validation = {}, []
    # Freeze feature definitions, candidates and threshold before viewing final test.
    for name, features in configurations.items():
        candidates = []
        for c in (0.01, 0.1, 1.0):
            fitted = classifier(features, c).fit(train, train.actual_up)
            brier = brier_score_loss(val.actual_up, fitted.predict_proba(val)[:, 1])
            candidates.append((float(brier), c, fitted))
        best_brier, best_c, selected = min(candidates, key=lambda x: (x[0], x[1]))
        models[name] = selected  # no refit on validation; final models see train only
        validation.append({"name": name, "selected_C": best_c, "brier": best_brier, "candidates": [{"C": c, "brier": score} for score, c, _ in candidates]})
    probabilities = {name: fitted.predict_proba(test)[:, 1] for name, fitted in models.items()}
    # Declared secondary comparison: identical rows, identical protocol, richer market inputs.
    extended_configurations = {"Market + BTC + intraday": MARKET_EXT,
                               "Market + BTC + intraday + X": MARKET_EXT + SOCIAL}
    ext_models, ext_validation = {}, []
    for name, features in extended_configurations.items():
        candidates = []
        for c in (0.01, 0.1, 1.0):
            fitted = classifier(features, c).fit(train, train.actual_up)
            brier = brier_score_loss(val.actual_up, fitted.predict_proba(val)[:, 1])
            candidates.append((float(brier), c, fitted))
        best_brier, best_c, selected = min(candidates, key=lambda x: (x[0], x[1]))
        ext_models[name] = selected
        ext_validation.append({"name": name, "selected_C": best_c, "brier": best_brier,
                               "candidates": [{"C": c, "brier": score} for score, c, _ in candidates]})
    ext_probabilities = {name: fitted.predict_proba(test)[:, 1] for name, fitted in ext_models.items()}
    EXT_BASE = "Market + BTC + intraday"
    EXT_X = "Market + BTC + intraday + X"
    majority = int(train.actual_up.mean() >= 0.5)
    metric_rows = [metric("Always up", test, np.ones(len(test))),
                   metric("Training majority", test, np.full(len(test), majority, dtype=float)),
                   metric("Previous direction", test, (test.ret1.to_numpy() > 0).astype(float))]
    metric_rows += [metric(name, test, p) for name, p in probabilities.items()]
    predictions = []
    for idx, row in enumerate(test.to_dict("records")):
        date = row["date"].strftime("%Y-%m-%d")
        prediction = {key: row[key] for key in ["coin", "actual_return", "actual_up", "post_count"]}
        prediction.update(date=date, cutoff=row["date"].isoformat(), target_end=(row["date"] + pd.Timedelta(days=1)).isoformat(),
                          market_prob=float(probabilities["Market only"][idx]), social_prob=float(probabilities["Market + X"][idx]),
                          attention_prob=float(probabilities["Market + attention"][idx]),
                          extended_market_prob=float(ext_probabilities[EXT_BASE][idx]),
                          extended_social_prob=float(ext_probabilities[EXT_X][idx]), persistence=int(row["ret1"] > 0),
                          examples=examples.get((row["coin"], date), []))
        prediction.update({key: float(row[key]) if row["post_count"] else None for key in ["sentiment", "bullish", "bearish"]})
        predictions.append(prediction)
    coverage = []
    for coin in COINS:
        source_mask = raw.tickers.apply(lambda x: coin in x if isinstance(x, (list, np.ndarray)) else False)
        selected = posts[posts.tickers.apply(lambda xs: coin in xs)]
        coverage.append({"coin": coin, "total_posts": int(source_mask.sum()), "original_posts": int((source_mask & raw.tweet_type.eq("tweet")).sum()),
                         "eligible_posts": int(len(selected)), "days_with_posts": int(selected.date.nunique()),
                         "price_days": int(len(prices[coin])), "first": str(prices[coin].date.min().date()), "last": str(prices[coin].date.max().date())})
    learning_curve = []
    for fraction in (0.25, 0.5, 1.0):
        dates = sorted(train.date.unique())
        small = train[train.date.isin(dates[-max(30, int(len(dates) * fraction)):])]
        c = next(x["selected_C"] for x in validation if x["name"] == "Market + X")
        fitted = classifier(MARKET + SOCIAL, c).fit(small, small.actual_up)
        learning_curve.append({"history_fraction": fraction, "train_dates": int(small.date.nunique()), "validation_brier": float(brier_score_loss(val.actual_up, fitted.predict_proba(val)[:, 1]))})
    mask = test.post_count.to_numpy() > 0
    subgroup = {"with_observed_posts": {"n": int(mask.sum()), "metrics": [metric(name, test[mask], p[mask]) for name, p in probabilities.items()] if mask.any() else []},
                "without_observed_posts": {"n": int((~mask).sum())}}
    extended = {
        "note": "Secondary comparison on the same held-out rows: does richer market context change the answer? Not used for selection.",
        "features": {"BTC": BTC, "intraday": INTRADAY},
        "validation": ext_validation,
        "metrics": [metric("Market only (primary)", test, probabilities["Market only"]),
                    metric("Market + X (primary)", test, probabilities["Market + X"])]
                   + [metric(name, test, p) for name, p in ext_probabilities.items()],
        "uncertainty": bootstrap_delta(test, ext_probabilities[EXT_BASE], ext_probabilities[EXT_X]),
        "primary_uncertainty": bootstrap_delta(test, probabilities["Market only"], probabilities["Market + X"]),
    }
    report = {
        "meta": {"project": "HypeCheck", "generated_at": datetime.now(timezone.utc).isoformat(),
                 "dataset": DATASET, "dataset_revision": DATASET_REV, "sentiment_model": MODEL, "sentiment_revision": MODEL_REV,
                 "disclaimer": "This project is a prototype and is not financial advice.",
                 "splits": {name: [part.date.min().strftime("%Y-%m-%d"), part.date.max().strftime("%Y-%m-%d")] for name, part in [("train", train), ("validation", val), ("test", test)]},
                 "evaluation_scope": "Retrospective frozen-model historical replay. No live forecasts, orders, or promised returns.",
                 "retrospective_pretrained_cutoff": START.strftime("%Y-%m-%d"),
                 "eligible_posts_missing_collection_timestamp": int(posts.first_posted_at.isna().sum()),
                 "availability_policy": "Original posts only; max(created_at, first_posted_at) where available. Prior-day posts only.",
                 "target": "Positive Binance USDT open-to-close return during the next UTC day; flat counts as not up.",
                 "hourly_market_source": "Binance Vision 1h spot klines from the pinned independent data package, aggregated only over UTC hours that closed before the cutoff.",
                 "btc_context_source": "BTC/USDT daily closes from the pinned independent market file, cross-checked bar-for-bar against our own Binance daily bars.",
                 "independent_scoring_revision": INDEPENDENT_MODEL_REV,
                 "limitations": [
                     "Selected influencers are not representative of all X. Missing observations do not mean no discussion occurred.",
                     "Sparse and changing coverage; author/source changes can mimic sentiment effects. No causal claim.",
                     "Pretrained sentiment checkpoint was public before the experiment period, but training-corpus overlap is not fully auditable.",
                     "English sentiment model; source language is not independently verified. Multi-coin posts share one sentiment score.",
                     "Sentiment is truncated to 128 tokens. Probabilities are not independently calibrated; sentiment certainty is not return certainty.",
                     "Historical USDT exchange bars exclude fees, slippage, latency and execution constraints; no profitability claim.",
                     "Only one held-out period. Date-block intervals are exploratory, and the final period must not be reused to tune models.",
                     "Two pinned revisions of the same sentiment checkpoint disagree on the direction of about 13% of shared posts, so these scores are model-revision dependent rather than a stable ground truth.",
                 ]},
        "metrics": metric_rows, "coverage": coverage, "predictions": predictions,
        "validation": validation, "learning_curve": learning_curve, "subgroups": subgroup,
        "label_counts": {name: {coin: {"n": int(len(part[part.coin.eq(coin)])), "up": int(part.loc[part.coin.eq(coin), "actual_up"].sum())} for coin in COINS} for name, part in [("train", train), ("validation", val), ("test", test)]},
        "uncertainty": bootstrap_delta(test, probabilities["Market only"], probabilities["Market + X"]),
        "extended": extended,
        "independent_check": independent_check(posts),
    }
    atomic_json(ROOT / "docs" / "report.json", report)
    frame.to_csv(ROOT / "data" / "features.csv", index=False)
    exports = {}
    for name, fitted in models.items():
        names = fitted.named_steps["prep"].get_feature_names_out().tolist()
        exports[name] = {"feature_names": names, "coefficients": fitted.named_steps["model"].coef_[0].tolist(),
                         "intercept": fitted.named_steps["model"].intercept_.tolist(),
                         "numeric_mean": fitted.named_steps["prep"].named_transformers_["numeric"].mean_.tolist(),
                         "numeric_scale": fitted.named_steps["prep"].named_transformers_["numeric"].scale_.tolist()}
    atomic_json(ROOT / "artifacts" / "coefficients.json", exports)
    files = [p for p in sorted((ROOT / "data").glob("*")) if p.is_file() and p.suffix in {".csv", ".parquet", ".json"}]
    files += [ROOT / "docs/report.json", ROOT / "artifacts/coefficients.json"]
    manifest = {"dataset_revision": DATASET_REV, "model_revision": MODEL_REV, "seed": 42, "code_sha256": digest(Path(__file__)),
                "files": {str(p.relative_to(ROOT)): digest(p) for p in files}}
    atomic_json(ROOT / "artifacts" / "manifest.json", manifest)
    for item in metric_rows:
        print(f"{item['name']}: accuracy={item['accuracy']:.4f}, balanced={item['balanced_accuracy']:.4f}, Brier={item['brier']:.4f}, n={item['n']}")
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--stage", choices=["data", "sentiment", "train", "all"], default="all")
    parser.add_argument("--batch-size", type=int, default=16)
    args = parser.parse_args()
    if not 1 <= args.batch_size <= 64:
        parser.error("batch-size must be between 1 and 64")
    raw, selected = load_posts()
    print(f"Source posts: {len(raw)}; eligible original deduplicated posts: {len(selected)}", flush=True)
    prices = {coin: load_prices(coin) for coin in COINS}
    hourly = {coin: load_hourly(coin) for coin in COINS}
    btc = load_btc()
    print("Price coverage:", {coin: len(p) for coin, p in prices.items()}, flush=True)
    print("Hourly coverage:", {coin: len(h) for coin, h in hourly.items()}, "BTC context rows:", len(btc), flush=True)
    if args.stage == "data":
        return
    scored = score_posts(selected, args.batch_size)
    if args.stage == "sentiment":
        return
    fit_and_report(raw, scored, prices, hourly, btc)


if __name__ == "__main__":
    main()

## 3 · Train and evaluate

This cell runs the whole experiment: fetch and validate, filter posts, score them with frozen
FinTwitBERT, build the lagged feature table, fit the models, and write `docs/report.json`.

CPU inference over ~1,100 posts is the slow part, and it checkpoints after every batch — if the cell
is interrupted, rerun it and it resumes from `data/sentiment.json` instead of starting over. A network
or validation failure stops the run; the pipeline never substitutes made-up data.


In [ ]:
!python pipeline.py --stage all --batch-size 16

## 4 · Results on held-out data

965 test coin-days across 193 shared UTC dates, 2026-04-12 → 2026-10-21. Every row above the line is
scored on exactly the same observations, so the comparison is like-for-like.


In [ ]:
import json
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:                                    # plain-python fallback
    def display(value):
        print(value)


report = json.loads(Path("docs/report.json").read_text())
print("Splits:", report["meta"]["splits"])

metrics = pd.DataFrame({k: v for k, v in row.items() if k != "per_coin"} for row in report["metrics"])
display(metrics[["name", "accuracy", "balanced_accuracy", "brier", "n", "n_dates"]].round(4))
print("Brier is a probability error — lower is better. Accuracy alone hides a near-50% problem.")

display(pd.DataFrame(report["coverage"]))

market = next(row for row in report["metrics"] if row["name"] == "Market only")["brier"]
social = next(row for row in report["metrics"] if row["name"] == "Market + X")["brier"]
uncertainty = report["uncertainty"]
print(f"\nMarket only Brier {market:.4f}  →  Market + X Brier {social:.4f}")
print(f"Social-minus-market Brier difference: {uncertainty['mean_social_minus_market_brier']:+.6f}"
      f"   (95% date-block interval {uncertainty['interval95'][0]:+.6f} to {uncertainty['interval95'][1]:+.6f})")
print("Positive means sentiment made probability error WORSE. The interval crosses zero, so the"
      "\nhonest reading is 'no demonstrated improvement', not 'proven harm'.")

print("\nSecondary arm — richer market context, identical held-out rows:")
display(pd.DataFrame(report["extended"]["metrics"])[["name", "accuracy", "brier", "n"]].round(4))

rows = []
for coin in ["DOGE", "SHIB", "PEPE", "BONK", "FLOKI"]:
    a = next(r for r in report["metrics"] if r["name"] == "Market only")["per_coin"][coin]
    b = next(r for r in report["metrics"] if r["name"] == "Market + X")["per_coin"][coin]
    rows.append({"coin": coin, "n": a["n"], "market_acc": round(a["accuracy"], 3), "social_acc": round(b["accuracy"], 3),
                 "market_brier": round(a["brier"], 4), "social_brier": round(b["brier"], 4)})
print("\nPer-coin detail:")
display(pd.DataFrame(rows))

print("\nDoes more training history help? (validation only, never test)")
display(pd.DataFrame(report["learning_curve"]))
print("Independent second scoring pass, different checkpoint revision:")
print(" ", report["independent_check"])

## 5 · Where it fails — the actually useful part

Two questions matter more than the headline number: does sentiment help where posts really exist, and
what do the confident mistakes look like?


In [ ]:
import json
from pathlib import Path

import pandas as pd

report = json.loads(Path("docs/report.json").read_text())
pred = pd.DataFrame(report["predictions"])
pred["up"] = pred.actual_up.astype(int)
pred["social_right"] = (pred.social_prob >= 0.5) == (pred.up == 1)
with_posts = pred[pred.post_count > 0]

print(f"Test coin-days with observed posts: {len(with_posts)} of {len(pred)} "
      f"({100 * len(with_posts) / len(pred):.1f}%). On the rest, market + X is a market model with a"
      "\nzero bolted on — which is why the pooled number flatters sentiment.")
for label, frame in (("all rows", pred), ("rows WITH posts", with_posts), ("rows WITHOUT posts", pred[pred.post_count == 0])):
    market = ((frame.market_prob >= 0.5) == (frame.up == 1)).mean()
    social = ((frame.social_prob >= 0.5) == (frame.up == 1)).mean()
    print(f"  {label:20s} n={len(frame):4d}   market-only {100 * market:5.2f}%   market + X {100 * social:5.2f}%")

disagree = pred[(pred.market_prob >= 0.5) != (pred.social_prob >= 0.5)]
pooled_market = int(((disagree.market_prob >= 0.5) == (disagree.up == 1)).sum())
pooled_social = int(((disagree.social_prob >= 0.5) == (disagree.up == 1)).sum())
print(f"\nThe two models disagreed on {len(disagree)} of {len(pred)} coin-days ({100 * len(disagree) / len(pred):.1f}%).")
print(f"  market-only right {pooled_market} · market + X right {pooled_social}")
posts_disagree = disagree[disagree.post_count > 0]
print(f"  on the {len(posts_disagree)} of those that had posts: market-only right "
      f"{int(((posts_disagree.market_prob >= 0.5) == (posts_disagree.up == 1)).sum())} · market + X right "
      f"{int(((posts_disagree.social_prob >= 0.5) == (posts_disagree.up == 1)).sum())}")

print("\nMost confident wrong calls from market + X, on coin-days that had posts:")
wrong = with_posts[~with_posts.social_right].copy()
wrong["confidence"] = (wrong.social_prob - 0.5).abs()
for row in wrong.sort_values("confidence", ascending=False).head(5).to_dict("records"):
    example = (row["examples"] or [{}])[0].get("text", "").replace("\n", " ")[:150]
    print(f"\n  {row['date']}  {row['coin']:5s}  p(up)={row['social_prob']:.3f}  actual next-day return "
          f"{row['actual_return']:+.2%}  sentiment {row['sentiment']:+.2f}  posts {row['post_count']}")
    print(f"     post: {example!r}")

print("\nReading these: the misses cluster on posts that are famous crypto-market commentary, whale"
      "\nwatching or sector talk that merely mentions a ticker — not a directional opinion about that"
      "\ncoin. The ticker tag comes from the archive's retrospective curation, so a bearish take on"
      "\n'the crypto gaming sector' became a bearish FLOKI score on a day FLOKI rose.")
print("\nThe model is also right for the wrong reason: daily prices are persistent, so 'yesterday's"
      "\ndirection' already scores 49.22% and the market features do most of the work.")

## 6 · The richer post-level model — who said it, how they said it, and why the 80/20 flatters it

Section 4 asks whether sentiment predicts direction. This section asks the question a judge should ask
next: **does more specific data help?** We rebuild the model on 40 features in seven blocks.

| Block | Features | n |
|---|---|---|
| Market | prior-day return 1/3/7, realised volatility, range, volume change | 6 |
| Past trend | trailing 10- and 30-day up-rate, trailing 10-day mean return | 3 |
| BTC and intraday | BTC prior-day 1/7, hourly volatility, hourly range, last-hour return, taker ratio | 6 |
| Post volume | posts per coin-day, log posts, distinct authors, top-author share | 4 |
| **The words used** | bullish, bearish, urgency, extreme claim, promotion, evidence, uncertainty, exclamations, external link, polarity, word count, CAPS share | 12 |
| **Who posted it** | author known, prior-30-day post count, prior-30-day active days | 3 |
| **Hype lexicon** | net bull-minus-bear word count, bull count, bear count, mean and max hype-meter score, share of posts matched | 6 |

Two limits, stated up front rather than buried, because each one costs a feature a judge might expect:

- **This data contains no follower counts.** The only reach proxy is engagement — views, likes, replies,
  retweets — and it exists on 177 of 1,274 posts, every one of them dated 2026-04-11 or later. Feeding it
  to the model would teach it *this row is from 2026*, not *this author is loud*. We exclude it, and the
  notebook prints that coverage so the exclusion is checkable rather than claimed.
- **64% of posts carry no author identity at all.** Author behaviour is therefore measured only where it
  exists, and flagged with `author_known`, so the learner can separate *no known author* from *an author
  with a zero*. The published table is anonymised: author keys are renumbered `author_001`, and post text,
  handles, display names, tweet IDs and per-person engagement are dropped before publication.

Then three splits, because here **the split decides the answer**:

| Split | Definition | What it hides |
|---|---|---|
| A | flat random 80/20 | 60.8% of its test rows share a calendar date with train, and 99.6% of its test posts come from authors already in train |
| B | chronological 80/20, 1-day purge | nothing — train on the past, test on the future |
| C | walk-forward, 6 date blocks | nothing — 4 forward windows, refit each time, 460 out-of-sample coin-days |

Two controls decide whether any gain is real: **label permutation** (destroy the training labels, keep
everything else) and a **feature placebo** (shuffle the post columns across rows, keep the labels).

### The hype meter — 234 curated words and phrases, measured against the corpus

You asked for something specific: label each post hype, bullish or bearish from its actual word choice,
name the words, and rate each one on a hype meter. `data/hype-lexicon.csv` is that artefact, and the
honest version of it is more interesting than a clean table would have been.

Method: curate the vocabulary (234 memecoin words, phrases and emojis), then **measure every entry against
the corpus** instead of trusting it. Two independent yardsticks:

- **bull / bear** — the share of posts containing the term that the *replication bundle's own* independent
  lexicon flagged as bullish or bearish.
- **hype meter (0–100)** — the share of posts containing the term that carry an independent hype marker:
  an exclamation mark, 30% or more capitals, a promotion flag, urgency, or an extreme claim. Corpus
  baseline: **16.1%**.

| Outcome of measurement | Terms |
|---|---|
| Validated by the corpus | 9 |
| Weak | 24 |
| Hype-only (direction-neutral) | 28 |
| Too rare to judge (n < 5) | 97 |
| **Never appears in the corpus** | **67** |
| Contradicted by the corpus | 9 |

Three findings that matter more than the table:

1. **The classic hype vocabulary is not in this data.** `to the moon`, `100x`, `rugpull`, `guaranteed`,
   `lambo`, `rekt` — 67 curated terms never appear once. This corpus is not retail shilling; it is
   technical-analysis commentary from 49 accounts.
2. **The hype meter does not measure hype.** Spearman between our post-level hype score and the
   independent hype markers is **+0.012** — zero. Every term scoring near 100 appears in a handful of
   posts, and each traces back to one account's ALL-CAPS ticker lists. In this corpus, hype lives in
   formatting, not vocabulary.
3. **The direction lexicon does work, moderately.** It agrees with the bundle's independent lexicon on
   **105 of the 145** posts where that lexicon fires (72%), Spearman on net bull-minus-bear intensity is
   **+0.41**, and it covers **30.8%** of posts against the bundle's **11.4%** — nearly three times the
   reach, from the same corpus.

Fed into the model as a seventh block, the lexicon is the first feature set to move the walk-forward AUC
clear of both controls: **0.5627** against a label-permutation null of 0.499 ± 0.023 (the real value sits
above all 20 draws) and a feature placebo of 0.514 ± 0.014. That is the strongest signal this project has
found — and it is still only AUC 0.56 across 460 out-of-sample coin-days, still beating its majority
baseline in 1 of 4 forward windows. Promising is not tradeable, and we are not going to pretend otherwise.

Measured by the next two cells:

| Split | Best block | Accuracy | Edge over majority class | AUC |
|---|---|---|---|---|
| A — flat 80/20 (leaky) | + the words used | 0.6014 | **+8.11pp** | 0.5995 |
| B — chronological 80/20 | + hype lexicon | 0.5751 | +6.36pp | 0.5849 |
| C — walk-forward | + FinTwitBERT sentiment | 0.5511 | — | 0.5687 |

The flat split is the one that would win a pitch, and it is the one that lies. **The best walk-forward
result sits 5.0 accuracy points below the best flat-split result**, because in a walk-forward the test rows
stop sharing dates and authors with training. Even in the honest split the richest model beats its own
majority baseline in only **2 of 4** forward windows. Author behaviour and the hype lexicon each add
roughly 2.5pp accuracy over the market-only model in the walk-forward (AUC 0.553 and 0.563 against 0.480)
— real movement, and still not a durable edge.

So the answer to *would more specific data help?* is the same as section 5's, arrived at a second way: not
as measured. What the extra data does buy is a demonstration of how a flat 80/20 split manufactures a
nine-point edge out of nothing, which is the failure mode this project exists to expose.


In [ ]:
%%writefile postmodel.py
"""HypeCheck: the richer post-level model.

Question: does adding author behaviour, the words used, hype markers and past
trends to the market features make next-day direction predictable?

Three splits, because the split you choose decides the answer:
  A  flat random 80/20          - the split everyone reaches for. Leaks: most test
                                  rows share a calendar date, and almost every test
                                  post comes from an author already in train.
  B  chronological 80/20        - train on the past, test on the future, 1-day purge.
  C  walk-forward, 6 date blocks - 4 forward windows, refit each time.

Two controls decide whether any gain is real:
  label permutation - destroy the training labels, keep everything else.
  feature placebo   - shuffle the post columns across rows, keep the labels.

Writes postmodel.json. Never fetches anything; reads the pinned files.
"""
import argparse, json
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FLAGS = ["urgency","extreme_claim","promotion","evidence_terms","uncertainty",
         "bullish_terms","bearish_terms","exclamations","external_link"]
MEANS = ["word_count","caps_share","prior_author_posts_30d","prior_author_active_days_30d"]

def load(data_dir):
    p = pd.read_csv(f"{data_dir}/post-features-anon.csv")
    p["date"] = pd.to_datetime(p.cutoff_utc, utc=True)
    key = "author_id" if "author_id" in p.columns else "author_key"
    # author_id is empty for unattributed posts; read_csv turns that into NaN
    p["auth"] = p[key].fillna("UNKNOWN").replace("", "UNKNOWN").astype(str)
    p["known"] = (p["auth"] != "UNKNOWN").astype(int)
    A = p.groupby(["coin","date"]).agg(
        n_posts=("coin","size"), n_authors=("auth","nunique"), author_known=("known","mean"),
        **{f"s_{c}": (c,"sum") for c in FLAGS},
        **{f"m_{c}": (c,"mean") for c in MEANS},
        lex_net=("lex_net","mean"), lex_bull=("lex_bull","mean"), lex_bear=("lex_bear","mean"),
        lex_hype=("lex_hype","mean"), lex_hype_max=("lex_hype","max"),
        lex_hit=("lex_net", lambda s: float((s != 0).mean()))).reset_index()
    for c in FLAGS:
        A[f"share_{c}"] = A[f"s_{c}"] / A["n_posts"]
    A["log_posts"] = np.log1p(A.n_posts)
    A["polarity"] = (A.s_bullish_terms - A.s_bearish_terms) / A.m_word_count.replace(0, np.nan)
    top = p.groupby(["coin","date","auth"]).size().groupby(level=[0,1]).max().rename("top_n").reset_index()
    A = A.merge(top, on=["coin","date"], how="left")
    A["top_author_share"] = A.top_n / A.n_posts

    f = pd.read_csv(f"{data_dir}/features.csv")
    f["date"] = pd.to_datetime(f.date, utc=True)
    f = f.loc[f.date >= pd.Timestamp("2024-03-01", tz="UTC")].sort_values(["coin","date"])
    f["trail_up10"]  = f.groupby("coin").actual_up.transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean())
    f["trail_up30"]  = f.groupby("coin").actual_up.transform(lambda s: s.shift(1).rolling(30, min_periods=5).mean())
    f["trail_ret10"] = f.groupby("coin").actual_return.transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean())
    M = A.merge(f, on=["coin","date"], how="inner").sort_values("date").reset_index(drop=True)

    # Author-behaviour means are undefined on days where every post is unattributed
    # (64% of rows carry no author). Impute 0 and keep author_known in the model so the
    # learner can tell "no known author" from "an author with a zero". polarity is also
    # undefined for empty text. Both are flagged in the write-up, not silently hidden.
    blocks = [c for c in ALLF + LEX + OURSENT if c in M.columns]
    M[blocks] = M[blocks].astype(float).fillna(0.0)
    return M, p

MK      = ["ret1","ret3","ret7","vol7","range1","volume_change"]
TREND   = ["trail_up10","trail_up30","trail_ret10"]
EXTRA   = ["btc_ret1","btc_ret7","hourly_vol1","hourly_range1","last_hour_ret","taker_ratio1"]
POSTVOL = ["log_posts","n_posts","n_authors","top_author_share"]
WORDS   = ["polarity","m_word_count","m_caps_share"] + [f"share_{c}" for c in FLAGS]
AUTHOR  = ["author_known","m_prior_author_posts_30d","m_prior_author_active_days_30d"]
LEX     = ["lex_net","lex_bull","lex_bear","lex_hype","lex_hype_max","lex_hit"]
OURSENT = ["sentiment","disagreement","bullish","bearish","post_count","has_posts"]
ALLF    = MK + TREND + EXTRA + POSTVOL + WORDS + AUTHOR

LADDER = [("market (6)", MK),
          ("+ past trend (3)", MK + TREND),
          ("+ BTC & intraday (6)", MK + TREND + EXTRA),
          ("+ post volume (4)", MK + TREND + EXTRA + POSTVOL),
          ("+ the words used (12)", MK + TREND + EXTRA + POSTVOL + WORDS),
          ("+ author behaviour (3)", ALLF),
          ("+ hype lexicon (6)", ALLF + LEX),
          ("+ FinTwitBERT sentiment (6)", ALLF + LEX + OURSENT)]

def model(kind, seed=0):
    if kind == "dummy":
        return DummyClassifier(strategy="most_frequent")
    if kind == "gb":
        return GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=seed)
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=4000))

def evaluate(tr, te, cols, kind, seed=0):
    m = model(kind, seed).fit(tr[cols].to_numpy(float), tr.actual_up.to_numpy(int))
    pr = m.predict_proba(te[cols].to_numpy(float))[:,1]
    y = te.actual_up.to_numpy(int)
    base = float(max(y.mean(), 1 - y.mean()))
    return dict(acc=float(accuracy_score(y, (pr > .5).astype(int))), base=base,
                edge=float(accuracy_score(y, (pr > .5).astype(int)) - base),
                brier=float(brier_score_loss(y, pr)),
                logloss=float(log_loss(y, np.clip(pr, 1e-9, 1 - 1e-9))),
                auc=float(roc_auc_score(y, pr)), n=int(len(y)))

def make_splits(M, blocks=6):
    dates = np.sort(M.date.unique())
    idx = np.arange(len(M))
    ti, ei = train_test_split(idx, test_size=0.2, random_state=20261114)
    flat = (M.iloc[ti], M.iloc[ei])
    cut = dates[int(round(len(dates) * 0.8))]
    chrono = (M.loc[M.date < (cut - pd.Timedelta(days=1))], M.loc[M.date >= cut])
    bnd = [dates[int(round(i * len(dates) / blocks))] for i in range(blocks)] + [dates[-1] + pd.Timedelta(days=1)]
    folds = []
    for i in range(1, blocks):
        lo, hi = bnd[i], bnd[i+1]
        tr = M.loc[M.date < (lo - pd.Timedelta(days=1))]
        te = M.loc[np.logical_and(M.date >= lo, M.date < hi)]
        if len(te) >= 20 and len(tr) >= 150:
            folds.append((tr, te))
    return flat, chrono, folds, cut

def show(tr, te, title, seeds_gb=3):
    base = float(max(te.actual_up.mean(), 1 - te.actual_up.mean()))
    print(f"\n=== {title}")
    print(f"    train {len(tr)} coin-days | test {len(te)} | test majority-class rate {base:.4f}")
    print(f"    {'feature block':30s}{'LOGIT acc':>11s}{'edge':>8s}{'AUC':>7s}{'Brier':>8s}"
          f"{'GBM acc':>9s}{'edge':>8s}{'AUC':>7s}{'Brier':>8s}")
    rows = {}
    for lab, cols in LADDER:
        rl = evaluate(tr, te, cols, "logit")
        g = [evaluate(tr, te, cols, "gb", s) for s in range(seeds_gb)]
        ga = float(np.mean([x["acc"] for x in g])); gu = float(np.mean([x["auc"] for x in g]))
        gb = float(np.mean([x["brier"] for x in g]))
        rows[lab] = dict(logit=rl, gbm_acc=ga, gbm_edge=ga-base, gbm_auc=gu, gbm_brier=gb)
        print(f"    {lab:30s}{rl['acc']:11.4f}{rl['edge']:+8.4f}{rl['auc']:7.4f}{rl['brier']:8.4f}"
              f"{ga:9.4f}{ga-base:+8.4f}{gu:7.4f}{gb:8.4f}")
    d = evaluate(tr, te, MK, "dummy")
    print(f"    {'majority baseline':30s}{d['acc']:11.4f}{0.0:+8.4f}{d['auc']:7.4f}{d['brier']:8.4f}")
    return rows

def show_walk(folds, seeds_gb=3):
    n = sum(len(t) for _, t in folds)
    print(f"\n=== SPLIT C - walk-forward: {len(folds)} forward windows, {n} out-of-sample coin-days")
    print(f"    {'feature block':30s}{'LOGIT acc':>11s}{'AUC':>7s}{'GBM acc':>9s}{'AUC':>7s}{'Brier':>8s}{'folds>base':>12s}")
    rows = {}
    for lab, cols in LADDER:
        la, lu, ga, gu, gb, win = [], [], [], [], [], 0
        for tr, te in folds:
            rl = evaluate(tr, te, cols, "logit")
            g = [evaluate(tr, te, cols, "gb", s) for s in range(seeds_gb)]
            a = float(np.mean([x["acc"] for x in g]))
            la.append(rl["acc"]); lu.append(rl["auc"]); ga.append(a)
            gu.append(float(np.mean([x["auc"] for x in g]))); gb.append(float(np.mean([x["brier"] for x in g])))
            if a > max(te.actual_up.mean(), 1 - te.actual_up.mean()):
                win += 1
        rows[lab] = dict(logit_acc=float(np.mean(la)), logit_auc=float(np.mean(lu)),
                         gbm_acc=float(np.mean(ga)), gbm_auc=float(np.mean(gu)),
                         gbm_brier=float(np.mean(gb)), folds_beating_base=win, folds=len(folds))
        print(f"    {lab:30s}{np.mean(la):11.4f}{np.mean(lu):7.4f}{np.mean(ga):9.4f}{np.mean(gu):7.4f}"
              f"{np.mean(gb):8.4f}{win:>9d}/{len(folds)}")
    return rows

def controls(folds, cols, label, perms=20, placebos=10, seed=0):
    rng = np.random.default_rng(seed)
    real = float(np.mean([evaluate(tr, te, cols, "gb", 0)["auc"] for tr, te in folds]))
    null = []
    for _ in range(perms):
        a = []
        for tr, te in folds:
            t2 = tr.copy(); t2["actual_up"] = rng.permutation(t2.actual_up.to_numpy())
            e2 = te.copy(); e2["actual_up"] = rng.permutation(e2.actual_up.to_numpy())
            a.append(evaluate(t2, e2, cols, "gb", 0)["auc"])
        null.append(float(np.mean(a)))
    null = np.array(null)
    plac = []
    for _ in range(placebos):
        a = []
        for tr, te in folds:
            t2 = tr.copy(); e2 = te.copy()
            for c in POSTVOL + WORDS + AUTHOR:
                v = t2[c].to_numpy().copy(); rng.shuffle(v); t2[c] = v
                v = e2[c].to_numpy().copy(); rng.shuffle(v); e2[c] = v
            a.append(evaluate(t2, e2, cols, "gb", 0)["auc"])
        plac.append(float(np.mean(a)))
    plac = np.array(plac)
    print(f"\n=== controls on the walk-forward ({label}, gradient boosting)")
    print(f"    real AUC                          {real:.4f}")
    print(f"    label-permuted null ({perms:2d} draws)   mean {null.mean():.4f}  sd {null.std():.4f}  max {null.max():.4f}  -> p = {(null >= real).mean():.2f}")
    print(f"    post-feature placebo ({placebos:2d} draws) mean {plac.mean():.4f}  sd {plac.std():.4f}")
    return dict(real_auc=real, null_mean=float(null.mean()), null_sd=float(null.std()),
                null_max=float(null.max()), p_value=float((null >= real).mean()),
                placebo_mean=float(plac.mean()), placebo_sd=float(plac.std()))

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", default="data")
    ap.add_argument("--json", default="postmodel.json")
    args = ap.parse_args()

    M, p = load(args.data_dir)
    print("=" * 100)
    print("HypeCheck - the richer post-level model")
    print(f"  coin-days with any post: {len(M)}  |  dates: {M.date.nunique()}  |  "
          f"{M.date.min().date()} -> {M.date.max().date()}  |  up-rate {M.actual_up.mean():.4f}")
    print(f"  posts: {len(p)}  |  distinct anonymised authors: {p.author_id.replace('', np.nan).nunique()}"
          f"  |  posts with a known author: {p.author_known.sum()} ({p.author_known.mean():.1%})")
    print(f"  feature blocks: market {len(MK)} + trend {len(TREND)} + btc/intraday {len(EXTRA)}"
          f" + post volume {len(POSTVOL)} + words {len(WORDS)} + author {len(AUTHOR)}"
          f" + hype lexicon {len(LEX)}  =  {len(ALLF)+len(LEX)} model features"
          f" (+{len(OURSENT)} FinTwitBERT in the last row)")

    flat, chrono, folds, cut = make_splits(M)
    ra = show(flat[0], flat[1], "SPLIT A - flat random 80/20 (the split to distrust)")
    shared = np.intersect1d(flat[1].date.unique(), flat[0].date.unique())
    au_tr = set(p.loc[p.date.isin(flat[0].date.unique()), "auth"])
    au_te = p.loc[p.date.isin(flat[1].date.unique()), "auth"]
    leak = dict(test_rows_sharing_a_date_with_train=float(flat[1].date.isin(shared).mean()),
                test_posts_from_authors_seen_in_train=float(au_te.isin(au_tr).mean()))
    print(f"    LEAKAGE: {leak['test_rows_sharing_a_date_with_train']:.1%} of test rows share a calendar date with train")
    print(f"    LEAKAGE: {leak['test_posts_from_authors_seen_in_train']:.1%} of test posts come from authors already in train")

    rb = show(chrono[0], chrono[1], f"SPLIT B - chronological 80/20, 1-day purge (cut {cut.date()})")
    rc = show_walk(folds)
    ctrl = controls(folds, ALLF + LEX, "all post+market features + hype lexicon")

    a_best = max(ra.items(), key=lambda t: t[1]["gbm_acc"])
    b_best = max(rb.items(), key=lambda t: t[1]["gbm_acc"])
    c_best = max(rc.items(), key=lambda t: t[1]["gbm_acc"])
    print("\n" + "=" * 100)
    print("READ THIS BEFORE QUOTING ANY NUMBER")
    print(f"  flat 80/20 (leaky)   best {a_best[0]:30s} GBM acc {a_best[1]['gbm_acc']:.4f}  edge {a_best[1]['gbm_edge']:+.4f}")
    print(f"  chronological 80/20  best {b_best[0]:30s} GBM acc {b_best[1]['gbm_acc']:.4f}  edge {b_best[1]['gbm_edge']:+.4f}")
    print(f"  walk-forward         best {c_best[0]:30s} GBM acc {c_best[1]['gbm_acc']:.4f}  "
          f"auc {c_best[1]['gbm_auc']:.4f}  folds beating base {c_best[1]['folds_beating_base']}/{c_best[1]['folds']}")
    print(f"  the flat split flatters every block by {a_best[1]['gbm_acc']-c_best[1]['gbm_acc']:+.4f} accuracy vs walk-forward,")
    print(f"  because {leak['test_rows_sharing_a_date_with_train']:.0%} of its test rows share a date with training rows.")
    print(f"  controls: real AUC {ctrl['real_auc']:.4f} vs label-permuted {ctrl['null_mean']:.4f} "
          f"(max {ctrl['null_max']:.4f}, p={ctrl['p_value']:.2f}) and feature-placebo {ctrl['placebo_mean']:.4f}.")
    print(f"  The best walk-forward block beats its majority baseline in "
          f"{c_best[1]['folds_beating_base']} of {c_best[1]['folds']} forward windows — real movement,"
          f" not yet a trading signal.")
    print("=" * 100)

    out = dict(rows=len(M), dates=int(M.date.nunique()), posts=len(p),
               posts_with_known_author=int(p.author_known.sum()),
               distinct_anonymised_authors=int(p.author_id.replace("", np.nan).nunique()),
               n_features=len(ALLF), n_features_with_sentiment=len(ALLF + OURSENT),
               split_a_flat=ra, split_b_chrono=rb, split_c_walkforward=rc,
               leakage=leak, controls=ctrl,
               best=dict(flat=a_best[0], chrono=b_best[0], walk=c_best[0]))
    with open(args.json, "w") as fh:
        json.dump(out, fh, indent=1)
    print(f"\nwrote {args.json}")

if __name__ == "__main__":
    main()

In [ ]:
!python postmodel.py --data-dir data --json postmodel.json

## 7 · Verification — the checks we ran against ourselves

A result that survives in only one direction is worthless, so the repository ships runnable gates. The
next two cells write `test_pipeline.py` and run it. These are adversarial tests, not decoration:

- **No future leakage.** Tripling prices from a chosen day onward must not change that day's market
  features *at all* — and must change the following day's, proving the test can detect a break.
- **Conservative availability.** `available_at = max(created_at, first_posted_at)`, and every retained
  post sits strictly inside the 24 hours before its own cutoff.
- **Tampering fails closed.** A corrupted `posts.parquet` raises on its SHA-256 instead of training.
- **No silent forward-filling.** Deleting one day of prices raises "Missing daily prices"; an
  out-of-range coin raises "Unknown coin".
- **Chronological purges.** The partition boundaries are asserted, including that no training target
  window reaches into validation.
- **Exported coefficients reproduce the report.** The plain-JSON parameters in
  `artifacts/coefficients.json` are re-scored against the published predictions, so the numbers are
  checkable without unpickling anything — we ship no pickles at all.

Beyond the Python gates, the dashboard's own DOM test renders the real `docs/report.json` through
`docs/app.js` and asserts both result tables, the coin and date controls, and that the reveal control
switches from hidden to revealed without altering a single reported probability. The published
dashboard reads the same `docs/report.json`, so the interface cannot drift from the measured results.


In [ ]:
%%writefile test_pipeline.py
"""Small runnable gates: .venv/bin/python -m unittest -v test_pipeline.py."""
import json
import tempfile
import unittest
from pathlib import Path
from unittest.mock import patch

import numpy as np
import pandas as pd

import pipeline as p


class PipelineChecks(unittest.TestCase):
    def test_market_features_cannot_see_today_or_future(self):
        prices = p.load_prices("DOGE")
        original = p.market_features(prices)
        changed = prices.copy()
        index = 80
        changed.loc[index:, ["open", "high", "low", "close", "volume"]] *= 3
        altered = p.market_features(changed)
        np.testing.assert_allclose(original.loc[index, p.MARKET].to_numpy(float), altered.loc[index, p.MARKET].to_numpy(float), rtol=0, atol=0)
        self.assertFalse(np.allclose(original.loc[index + 1, p.MARKET].to_numpy(float), altered.loc[index + 1, p.MARKET].to_numpy(float)))
        np.testing.assert_allclose(original.actual_return, prices.close / prices.open - 1)

    def test_source_availability_is_conservative(self):
        raw, selected = p.load_posts()
        self.assertFalse(raw.tweet_id.duplicated().any())
        self.assertTrue(selected.tweet_type.eq("tweet").all())
        self.assertTrue((selected.available_at >= selected.created_at).all())
        observed = selected.first_posted_at.notna()
        self.assertTrue((selected.loc[observed, "available_at"] >= selected.loc[observed, "first_posted_at"]).all())
        self.assertTrue((selected.available_at < selected.date).all())
        self.assertTrue((selected.available_at >= selected.date - pd.Timedelta(days=1)).all())
        self.assertFalse(selected.text.duplicated().any())
        self.assertTrue((selected.available_at >= p.START - pd.Timedelta(days=1)).all())

    def test_pinned_dataset_tampering_fails_closed(self):
        with tempfile.TemporaryDirectory() as folder:
            root = Path(folder)
            (root / "data").mkdir()
            (root / "data/posts.parquet").write_bytes(b"PAR1not the pinned dataset")
            with patch.object(p, "ROOT", root):
                with self.assertRaisesRegex(ValueError, "SHA-256"):
                    p.load_posts()

    def test_missing_price_day_is_not_forward_filled(self):
        prices = p.load_prices("DOGE").drop(index=5)
        with tempfile.TemporaryDirectory() as folder:
            root = Path(folder)
            (root / "data").mkdir()
            prices.to_csv(root / "data/DOGE-prices.csv", index=False)
            with patch.object(p, "ROOT", root):
                with self.assertRaisesRegex(ValueError, "Missing daily prices"):
                    p.load_prices("DOGE")
        with self.assertRaisesRegex(ValueError, "Unknown coin"):
            p.load_prices("../../secret")

    def test_split_boundaries_and_complete_coin_support(self):
        frame = pd.read_csv(p.ROOT / "data/features.csv")
        frame["date"] = pd.to_datetime(frame.date, utc=True)
        train, val, test = p.split_dates(frame)
        self.assertLess(train.date.max() + pd.Timedelta(days=1), val.date.min())
        self.assertLess(val.date.max() + pd.Timedelta(days=1), test.date.min())
        for a, b in [(train, val), (val, test), (train, test)]:
            self.assertFalse(set(a.date) & set(b.date))
        for part in (train, val, test):
            self.assertEqual(set(part.coin), set(p.COINS))
            self.assertTrue((part.groupby("date").coin.nunique() == 5).all())
            self.assertEqual(part.actual_up.nunique(), 2)
        self.assertTrue((frame.loc[frame.post_count.eq(0), p.SOCIAL] == 0).all().all())

    def test_report_metrics_match_saved_predictions(self):
        report = json.loads((p.ROOT / "docs/report.json").read_text())
        rows = pd.DataFrame(report["predictions"])
        self.assertEqual(len(rows), len(rows.drop_duplicates(["coin", "date"])))
        self.assertEqual(set(rows.coin), set(p.COINS))
        for metric in report["metrics"]:
            self.assertEqual(metric["n"], len(rows))
            self.assertEqual(metric["n_dates"], rows.date.nunique())
        for name, column in [("Market only", "market_prob"), ("Market + attention", "attention_prob"), ("Market + X", "social_prob")]:
            measured = p.scores(rows.actual_up, rows[column])
            saved = next(x for x in report["metrics"] if x["name"] == name)
            for key in measured:
                self.assertAlmostEqual(measured[key], saved[key], places=12)
        for row in report["predictions"]:
            cutoff = pd.Timestamp(row["cutoff"])
            self.assertEqual(pd.Timestamp(row["target_end"]) - cutoff, pd.Timedelta(days=1))
            self.assertEqual(row["actual_up"], int(row["actual_return"] > 0))
            self.assertEqual(row["sentiment"] is None, row["post_count"] == 0)
            for post in row["examples"]:
                available = pd.Timestamp(post["available_at"])
                self.assertLess(available, cutoff)
                self.assertGreaterEqual(available, cutoff - pd.Timedelta(days=1))

    def test_exported_json_coefficients_reproduce_predictions(self):
        exports = json.loads((p.ROOT / "artifacts/coefficients.json").read_text())
        report = json.loads((p.ROOT / "docs/report.json").read_text())
        frame = pd.read_csv(p.ROOT / "data/features.csv")
        frame["date"] = pd.to_datetime(frame.date, utc=True)
        _, _, test = p.split_dates(frame)
        for name, column in [("Market only", "market_prob"), ("Market + attention", "attention_prob"), ("Market + X", "social_prob")]:
            artifact = exports[name]
            names = artifact["feature_names"]
            numeric = [key.removeprefix("numeric__") for key in names if key.startswith("numeric__")]
            matrix = (test[numeric].to_numpy() - artifact["numeric_mean"]) / artifact["numeric_scale"]
            categories = [key.removeprefix("coin__coin_") for key in names if key.startswith("coin__")]
            onehot = np.column_stack([test.coin.eq(coin).to_numpy(float) for coin in categories])
            logits = np.column_stack([matrix, onehot]) @ artifact["coefficients"] + artifact["intercept"][0]
            prob = 1 / (1 + np.exp(-logits))
            np.testing.assert_allclose(prob, [r[column] for r in report["predictions"]], atol=1e-12, rtol=1e-12)

    def test_manifest_binds_code_inputs_and_published_outputs(self):
        manifest = json.loads((p.ROOT / "artifacts/manifest.json").read_text())
        self.assertEqual(manifest["code_sha256"], p.digest(p.ROOT / "pipeline.py"))
        for key in ["docs/report.json", "artifacts/coefficients.json"]:
            self.assertIn(key, manifest["files"])
        for filename, checksum in manifest["files"].items():
            self.assertEqual(checksum, p.digest(p.ROOT / filename))

    def test_hourly_features_cannot_see_the_cutoff_day(self):
        hourly = p.load_hourly("DOGE")
        bars = pd.read_csv(p.ROOT / "data/market-hourly.csv")
        bars = bars[bars.coin.eq("DOGE")].copy()
        bars["open_time"] = pd.to_datetime(bars.open_time, utc=True)
        prior = bars[bars.open_time.dt.date == pd.Timestamp("2024-04-10").date()].sort_values("open_time")
        cutoff_day = hourly[hourly.date.eq(pd.Timestamp("2024-04-11", tz="UTC"))].iloc[0]
        self.assertAlmostEqual(cutoff_day.hourly_vol1, np.log(prior.close_usdt / prior.open_usdt).std(ddof=1), places=12)
        self.assertAlmostEqual(cutoff_day.last_hour_ret, np.log(prior.close_usdt.iloc[-1] / prior.open_usdt.iloc[-1]), places=12)
        self.assertAlmostEqual(cutoff_day.taker_ratio1, prior.taker_buy_volume_usdt.sum() / prior.volume_usdt.sum(), places=12)
        self.assertEqual(int(prior.open_time.dt.hour.max()), 23)
        # The cutoff day's own hours must not appear anywhere in its features.
        following = bars[bars.open_time.dt.date == pd.Timestamp("2024-04-11").date()]
        self.assertNotAlmostEqual(cutoff_day.hourly_range1, (following.high_usdt.max() - following.low_usdt.min()) / following.close_usdt.iloc[-1], places=6)

    def test_hourly_gap_fails_closed(self):
        with tempfile.TemporaryDirectory() as folder:
            root = Path(folder)
            (root / "data").mkdir()
            bars = pd.read_csv(p.ROOT / "data/market-hourly.csv")
            bars = bars[bars.coin.eq("DOGE")].reset_index(drop=True).drop(index=100)
            bars.to_csv(root / "data/market-hourly.csv", index=False)
            with patch.object(p, "ROOT", root), patch.object(p, "HOURLY_SHA256", p.digest(root / "data/market-hourly.csv")):
                with self.assertRaisesRegex(ValueError, "contiguous"):
                    p.load_hourly("DOGE")

    def test_pinned_secondary_sources_fail_closed(self):
        for name, attribute, loader in [("market-hourly.csv", "HOURLY_SHA256", "load_hourly"),
                                        ("expanded-daily.csv", "EXPANDED_SHA256", "load_btc")]:
            with tempfile.TemporaryDirectory() as folder:
                root = Path(folder)
                (root / "data").mkdir()
                (root / "data" / name).write_text("coin,tampered\n")
                with patch.object(p, "ROOT", root), patch.object(p, attribute, "0" * 64):
                    with self.assertRaisesRegex(ValueError, "SHA-256"):
                        p.load_hourly("DOGE") if loader == "load_hourly" else p.load_btc()

    def test_btc_context_is_lagged_and_matches_our_own_bars(self):
        btc = p.load_btc()
        self.assertGreater(len(btc), 1000)
        self.assertTrue((btc.date.diff().dropna() == pd.Timedelta(days=1)).all())
        tail = btc.dropna(subset=["btc_ret1", "btc_ret7"])
        self.assertTrue(np.isfinite(tail[["btc_ret1", "btc_ret7"]].to_numpy(float)).all())
        # The independent daily file must reproduce our own Binance bars exactly (checked inside load_btc).
        ours = p.load_prices("DOGE")
        merged = btc.merge(ours[["date", "close"]], on="date", how="inner")
        self.assertGreater(len(merged), 900)

    def test_extended_market_features_are_lagged_too(self):
        prices = p.load_prices("DOGE")
        frame = pd.read_csv(p.ROOT / "data/features.csv")
        frame["date"] = pd.to_datetime(frame.date, utc=True)
        self.assertTrue(np.isfinite(frame[p.MARKET_EXT].to_numpy(float)).all())
        del prices
        for coin in p.COINS:
            hourly = p.load_hourly(coin)
            joined = frame[frame.coin.eq(coin)].merge(hourly, on="date", how="inner", suffixes=("", "_source"))
            self.assertEqual(len(joined), 965)
            for column in p.INTRADAY:
                np.testing.assert_allclose(joined[column].to_numpy(float), joined[column + "_source"].to_numpy(float), rtol=0, atol=0)
            del joined, hourly

    def test_primary_metrics_are_unchanged_by_the_extension(self):
        report = json.loads((p.ROOT / "docs/report.json").read_text())
        published = {"Always up": 0.465284974093264, "Training majority": 0.534715025906736,
                     "Previous direction": 0.492227979274611, "Market only": 0.536787564766839,
                     "Market + attention": 0.544041450777202, "Market + X": 0.530569948186528}
        measured = {row["name"]: row["accuracy"] for row in report["metrics"]}
        self.assertEqual(set(measured), set(published))
        for name, value in published.items():
            self.assertAlmostEqual(measured[name], value, places=12)

    def test_extended_metrics_match_saved_predictions(self):
        report = json.loads((p.ROOT / "docs/report.json").read_text())
        rows = pd.DataFrame(report["predictions"])
        pairs = [("Market + BTC + intraday", "extended_market_prob"),
                 ("Market + BTC + intraday + X", "extended_social_prob")]
        for name, column in pairs:
            measured = p.scores(rows.actual_up, rows[column])
            saved = next(x for x in report["extended"]["metrics"] if x["name"] == name)
            for key in measured:
                self.assertAlmostEqual(measured[key], saved[key], places=12)

    def test_independent_scoring_check_is_reproducible(self):
        report = json.loads((p.ROOT / "docs/report.json").read_text())
        checked = report["independent_check"]
        self.assertTrue(checked["available"])
        self.assertNotEqual(checked["our_model_revision"], checked["independent_model_revision"])
        self.assertEqual(checked["shared_posts"], 1122)
        cache = json.loads((p.ROOT / "data/sentiment.json").read_text())["scores"]
        other = pd.read_csv(p.ROOT / "data/independent-post-scores.csv", usecols=["post_id", "p_bullish"])
        other["post_id"] = other.post_id.astype(str)
        merged = other.drop_duplicates("post_id")
        merged = merged[merged.post_id.isin([str(k) for k in cache])]
        delta = np.abs([cache[k]["bullish"] - merged.loc[merged.post_id.eq(str(k)), "p_bullish"].iloc[0] for k in merged.post_id])
        self.assertAlmostEqual(float(delta.mean()), checked["mean_abs_p_bullish_difference"], places=12)

    def test_empty_subgroup_and_invalid_probabilities(self):
        self.assertEqual(p.scores([], [])["n"], 0)
        self.assertIsNone(p.scores([], [])["accuracy"])
        for values in ([np.nan], [-0.1], [1.1]):
            with self.assertRaisesRegex(ValueError, "Invalid forecast"):
                p.scores([1], values)


if __name__ == "__main__":
    unittest.main()

In [ ]:
!python -m unittest -v test_pipeline.py

## 8 · Write-up

**HypeCheck: Signal or Noise?**
**Track:** AI + Finance · **Team:** Nathaniel Wong (model), Aaden Yick (data), Alastair Chu (demo and write-up) — Malvern College Hong Kong

*This project is a prototype and is not financial advice.*

### The problem
A beginner crypto investor watching DOGE, SHIB, PEPE, BONK and FLOKI sees a timeline full of confident
posts and has no way to tell whether any of it is information or noise. The stakes are real: memecoin
moves are violent, and "the crowd knows something" is exactly the belief that hype monetises. We built
the tool we wanted as sceptics — one that answers *"does this actually predict anything?"* with a
measured number instead of a promise.

The guide's example of a well-scoped version of a too-broad idea is *"a test of whether news-headline sentiment helps predict next-day price moves for five stocks, reported honestly either way."* Our project is that test run on five memecoins and X posts rather than headlines, and we report the answer either way. It came out negative: sentiment did not improve the forecasts.

### What we built
In: the posts published in the 24 hours before a fixed UTC cutoff, plus that coin's own price history.
Out: a probability that the next UTC day closes up, next to three simple baselines and a market-only
model, with the option to replay any held-out date and see what happened. It runs as this notebook and
as a static dashboard ([https://adeviwon.github.io/hypecheck/](https://adeviwon.github.io/hypecheck/)) reading the same report.

### What we added beyond suggested project 4.2.4
The brief prescribes headlines → sentiment → predict the Dow. We changed the market (five memecoins),
the news source (X posts) and the evaluation discipline: a pre-declared selection rule, purged
partitions, an ablation, a date-block interval, a second independent scoring pass, and per-coin
sparse-coverage accounting. We did **not** invent FinTwitBERT or logistic regression, and we say so.

### How it works
- **Data.** `StephanAkkerman/FinTwitBERT-sentiment` is the model; the data is `StephanAkkerman/financial-tweets-crypto` at
  revision `c4b17a94a667af3da1c9ed5144f998e3a4f29966` — 104,379 posts reduced to **1,124 eligible** original deduplicated posts.
  Prices: continuous Binance USDT daily bars, 1,025 days per coin, 2024-01-01 → 2026-10-21.
- **Sentiment.** Frozen FinTwitBERT (`da059da3b3bbcb43f9ed1aeb5ae61644010c7e1e`), bullish / bearish / neutral, 128-token truncation,
  no fine-tuning of any kind.
- **Model.** Pooled regularised logistic regression on lagged market features plus a one-hot coin
  identity, `C` selected on validation Brier, decision threshold 0.5.

### Results
| Method | Direction accuracy | Balanced accuracy | Brier error (lower is better) | Test coin-days |
|---|---|---|---|---|
| Always up | 46.53% | 50.00% | 0.5347 | 965 |
| Training majority | 53.47% | 50.00% | 0.4653 | 965 |
| Previous direction | 49.22% | 48.95% | 0.5078 | 965 |
| Market only | 53.68% | 51.50% | 0.2481 | 965 |
| Market + attention | 54.40% | 52.23% | 0.2480 | 965 |
| Market + X | 53.06% | 51.23% | 0.2491 | 965 |

Social-minus-market Brier difference **+0.001065**, 95% date-block interval **[-0.000159, +0.002483]** (1,000
circular 7-day moving-block resamples with coins kept together by date). Positive favours market-only
and the interval crosses zero, so **we do not demonstrate that sentiment improves these forecasts.**

The sharper finding is where sentiment should have helped most. On the **142 test coin-days that
actually had posts**, market-only scored 51.41% direction accuracy against 42.96% for market + X. On
the 92 coin-days where the two models disagreed, market-only was right 49 times against
43 — and on the 78 of those with posts, 45 against 33.

Secondary arm, identical held-out rows:

| Method (secondary arm, identical held-out rows) | Direction accuracy | Brier error |
|---|---|---|
| Market only (primary) | 53.68% | 0.2481 |
| Market + X (primary) | 53.06% | 0.2491 |
| Market + BTC + intraday | 50.26% | 0.2495 |
| Market + BTC + intraday + X | 49.53% | 0.2504 |

Coverage, which is the real constraint:

| Coin | Eligible original posts | Days with posts | Price days |
|---|---|---|---|
| DOGE | 638 | 306 | 1025 |
| SHIB | 45 | 42 | 1025 |
| PEPE | 244 | 167 | 1025 |
| BONK | 137 | 94 | 1025 |
| FLOKI | 150 | 114 | 1025 |

### Where it fails
- Posts that are sector commentary, whale-watching or market-wide news but carry a ticker get read as
  coin sentiment. Our most confident error, 2026-10-10 DOGE (`p(up)=0.33`, actual **+0.45%**), rests on
  a post about Bitcoin liquidations; 2026-06-15 FLOKI (`p(up)=0.36`, actual **+3.14%**) rests on "is the
  crypto gaming sector over?" — a bearish sector take, not a call on FLOKI.
- Sparse coverage is structural: DOGE has 638 eligible posts across 306 days while SHIB has 45 across
  42. 142 of 965 test coin-days had no eligible post at all.
- The archive's ticker tags are retrospective curation, so we cannot prove the tag was right at the time.
- Sentiment here means "text about this coin", never "this coin will rise".

### Would better data fix this?
The obvious objection is that our source is thin. We tested it directly **after** the run was frozen —
nothing below changed the model, the features or the selection, and we report it as a post-hoc diagnostic.

Coverage is the binding constraint, and raising it moves the headline the wrong way:

| Test coin-days | n | Market only | Market + X |
|---|---:|---:|---:|
| Had eligible posts | 142 | 51.41% | **42.96%** |
| No eligible posts | 823 | 54.07% | 54.80% |

Pooled: `0.147 x 42.96 + 0.853 x 54.80 = 53.06%` — exactly the figure above. Sentiment exists on 14.7% of
test days and lowers accuracy on precisely those days, so giving every day posts would push the pooled
number toward **43%**, not up.

More posts per day, and stronger sentiment, behave the same way:

| Slice of the 142 rows with posts | n | Market only | Market + X |
|---|---:|---:|---:|
| Exactly 1 post | 122 | 52.5% | 44.3% |
| 2 posts | 11 | 54.5% | 36.4% |
| \|sentiment\| 0–0.25 | 16 | 68.8% | 37.5% |
| \|sentiment\| 0.25–0.6 | 25 | 56.0% | 44.0% |
| \|sentiment\| 0.6–1.0 | 101 | 47.5% | 43.6% |

There is no monotone relationship for more data to sharpen — Spearman rho of sentiment against next-day
return: train **+0.076** (p=0.070, n=549), validation **−0.180** (p=0.335, n=30), test **−0.027** (p=0.733,
n=143). One relationship did reach p<0.05: |sentiment| against next-day high-low range on the test set
(rho +0.196). It flipped sign on train (−0.064) and validation (−0.201), so we treat it as a false positive
rather than a lead, and we are flagging it here rather than quietly keeping the one that looked good.

The honest reading: **this is not a data-volume problem.** Next-day direction of a liquid, heavily watched
asset is close to what public posts cannot tell you, and a larger archive of the same kind of posts would
sharpen a number that already sits next to a coin flip.

One calibration we owe a reader. On direction accuracy, market-only beats the trivial "always predict the
training majority" rule by **0.21 points** (53.68% vs 53.47%). Where it genuinely earns its keep is
probability error: Brier **0.2481** against the same baseline's 0.4653 — **46.7% lower**. The model's value
is calibrated uncertainty, not direction, and the interface says so on its face.

### What we'd do next
1. Collect a prospective, source-balanced stream instead of a retrospective archive, and evaluate on a
   genuinely later period, with the sentiment model fine-tuned on memecoin language.
2. Label a few hundred posts by hand for *intent* — directional call, sector news, promotion, factual —
   rather than trusting the archive's retrospective ticker tags, then use those labels to score posts
   Our failure analysis says that is where most of the error lives.
3. Report a cost-sensitive measure, because a missed move and a false alarm cost a beginner very
   differently.

### Limitations
- Selected influencers are not representative of all X. Missing observations do not mean no discussion occurred.
- Sparse and changing coverage; author/source changes can mimic sentiment effects. No causal claim.
- Pretrained sentiment checkpoint was public before the experiment period, but training-corpus overlap is not fully auditable.
- English sentiment model; source language is not independently verified. Multi-coin posts share one sentiment score.
- Sentiment is truncated to 128 tokens. Probabilities are not independently calibrated; sentiment certainty is not return certainty.
- Historical USDT exchange bars exclude fees, slippage, latency and execution constraints; no profitability claim.
- Only one held-out period. Date-block intervals are exploratory, and the final period must not be reused to tune models.
- Two pinned revisions of the same sentiment checkpoint disagree on the direction of about 13% of shared posts, so these scores are model-revision dependent rather than a stable ground truth.


## 9 · Submission form answers

| Form question | Our answer |
|---|---|
| **Project title** | HypeCheck: Signal or Noise? |
| **Building as** | Team |
| **Team details** | Nathaniel Wong (model) · Aaden Yick (data) · Alastair Chu (demo and write-up) — Malvern College Hong Kong. *Before submitting, paste each member's email here in that order, spelled as it should appear on certificates.* |
| **Track** | AI + Finance |
| **Starting point** | 4.2.4 News headlines and the stock market |
| **Scope, in the guide's format** | A test of whether X-post sentiment helps predict next-day price moves for five memecoins (DOGE, SHIB, PEPE, BONK, FLOKI), reported honestly either way. |
| **One-sentence pitch** | HypeCheck takes a memecoin's X posts and its own price history and tells a beginner investor whether the hype carries any predictive signal at all — reporting honestly that, in our measured run, it does not. |
| **Code link** | [This Colab notebook](https://colab.research.google.com/github/adeviwon/hypecheck/blob/main/HypeCheck.ipynb) or the [repository](https://github.com/adeviwon/hypecheck) |
| **Headline result** | **54.40%** accuracy against a **53.47%** base rate on 965 held-out coin-days — a +0.93 point edge. A richer 40-feature model reaches **60.14%** on a flat random 80/20 split, but 60.8% of that split's test rows share a calendar date with training rows and 99.6% of its test posts come from authors already seen in training; the same model walk-forward gives **55.11%** and beats its baseline in 2 of 4 windows. We report the split that survives, not the one that flatters. |
| **Write-up** | Section 8 of this notebook, and the repository README |
| **Demo video link** | *optional — add the unlisted YouTube or Drive link* |
| **What we added** | A different market (five memecoins, not the Dow) and source (X posts, not headlines); a frozen pretrained sentiment model with conservative availability alignment; an ablation from market-only → +attention → +sentiment; a pre-declared selection rule with purged partitions; a date-block uncertainty estimate; a second independent scoring pass; a failure analysis of the confident misses; a 40-feature post-level model run across three split protocols that catches an ordinary random split manufacturing a 60.14% result and reports the honest walk-forward number instead; and a 234-term hype lexicon with a measured per-word hype meter. |
| **Datasets, models and AI tools used** | HF `StephanAkkerman/financial-tweets-crypto`; HF `StephanAkkerman/FinTwitBERT-sentiment`; Binance public klines; scikit-learn, pandas, NumPy, PyArrow, Requests, PyTorch, Hugging Face Transformers, safetensors; Hermes Agent / OpenAI coding assistance. Full credits in section 10. |
| **New to AI?** | No — at least one of us has built with AI before |

*Prototype, not financial advice. The results are the ones we actually measured, including the negative one.*


## 10 · Sources and credits

- **Dataset:** [StephanAkkerman/financial-tweets-crypto](https://huggingface.co/datasets/StephanAkkerman/financial-tweets-crypto) (MIT) — revision `c4b17a94a667af3da1c9ed5144f998e3a4f29966`, upstream LFS SHA-256 `81b2f7563a44c1a103e9468a83d52319dfeae0fc616422b4a4df90d8d684b960`.
- **Pretrained model:** [StephanAkkerman/FinTwitBERT-sentiment](https://huggingface.co/StephanAkkerman/FinTwitBERT-sentiment) (MIT), Stephan Akkerman and Tim Koornstra. Reused, not our invention; the independent check uses revision `c079da7618e551d182ffdf68ef7aa25cb9a79667`.
- **Hourly market bars and BTC context:** Binance Vision spot klines, supplied in an independent HypeCheck data package; attributed to Binance Vision under its dataset terms and CC BY-NC-SA 4.0. The daily rows were cross-checked bar-for-bar against our own Binance bars and matched exactly.
- **Market data and field definitions:** [Binance spot API documentation](https://github.com/binance/binance-spot-api-docs/blob/master/rest-api.md#klinecandlestick-data). Binance market data is not relicensed as our code.
- **Libraries:** PyTorch, Hugging Face Transformers, safetensors, scikit-learn, pandas, NumPy, PyArrow, Requests — each keeps its own licence.
- **Evaluation guidance:** [scikit-learn common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html).
- **Hackathon brief and rules:** [highschoolhackathon.org](https://highschoolhackathon.org/briefs.html) and the participant guide, sections 5–10.
- **AI tools:** Hermes Agent with OpenAI models was used for research, implementation, review and testing. Every number here is reproducible from this notebook, and the team can explain every part.

Code is MIT licensed (`LICENSE`). Third-party posts, model weights, market data and libraries keep
their original terms.

### Files this notebook produces
`docs/report.json` (every held-out prediction, metric, split, coverage figure and uncertainty
estimate), `artifacts/coefficients.json` (plain JSON model parameters — no pickles),
`data/features.csv` (the exact modelling table) and `artifacts/manifest.json` (SHA-256 of every input
and output).

### Data files this repository publishes, with their digests

| File | Size | SHA-256 (first 16 characters) |
|---|---|---|
| `posts.parquet` | 20.74 MB | `81b2f7563a44c1a1...` |
| `market-hourly.csv` | 20.67 MB | `d6819bf4ff46c31c...` |
| `expanded-daily.csv` | 0.90 MB | `2aab9bbe2c0d6782...` |
| `independent-post-scores.csv` | 0.27 MB | `bb35778b88edeb9e...` |
| `DOGE-prices.csv` | 0.07 MB | `afc23a84c9582791...` |
| `SHIB-prices.csv` | 0.08 MB | `6f65ac812bbcbde7...` |
| `PEPE-prices.csv` | 0.08 MB | `024ecb4a9da72818...` |
| `BONK-prices.csv` | 0.08 MB | `18be2016111fa5fc...` |
| `FLOKI-prices.csv` | 0.09 MB | `1348af136631c9b4...` |
| `post-features-anon.csv` | 0.12 MB | `2d71dbc5da4cd3ef...` |
| `hype-lexicon.csv` | 0.01 MB | `598d46cb09ddba85...` |

`posts.parquet` is fetched from Hugging Face at the pinned revision; the rest are the exact bytes our
measured run used, hosted in the [repository](https://github.com/adeviwon/hypecheck) and verified against these
digests on every run.


In [ ]:
# 11 · Take the results with you (optional; Colab only).
if IN_COLAB:
    from google.colab import files

    files.download("docs/report.json")
    files.download("data/features.csv")
    files.download("artifacts/coefficients.json")
else:
    print("Not in Colab — results are on disk at docs/report.json, data/features.csv, artifacts/.")